# LoRA + Tabular Complements

This ensemble combines language-model and encoder readouts, adaptive centered stylegram evidence, a geometric TabPFN pool and a small TabICL complement. It builds on [Zbigniew Kirchner's adaptive-centering and Claude-gate ensemble](https://www.kaggle.com/code/zbigniewkirchner/0-552-adaptive-centering-lora-claude-gates), which extends [Lucas Boesen's LoRA ensemble](https://www.kaggle.com/code/lucasboesen/lora-x-adaptive-centering). The original component weights, all three Qwen LoRA branches, encoder and token-attention heads, and within-Claude specialists are retained.

The sparse stylegram component follows [Obstacle Developer's centered-stylegram recipe](https://www.kaggle.com/code/obstacledeveloper/local-candidate-adaptive-centered-stylegram). A second LinearSVC is trained on features centered within training prompts. Query groups use text-only, median-normalized TF-IDF distances and average-linkage clustering at 0.90. Centered evidence replaces plain stylegram evidence only in groups of 2–18 passages, retaining the original component weight 1.516.

Our tabular representation reuses the two frozen feature views already extracted by the reference model: two 64-coordinate PCA views, 24 fixed text statistics, and two 16-coordinate masked TF-IDF/SVD views. All fitted transforms use training rows. TabPFN 3.5 uses the standard checkpoint and 128 estimators.

The original 50-step baseline Sinkhorn pass uses strength 0.25. Given its class-aligned posterior P and the raw TabPFN probabilities Q, the complement uses a normalized geometric pool G: each row is proportional to P^0.8 times Q^0.2, with probabilities floored at 1e-300 before taking logarithms. These values are exponents, not arithmetic probability weights. The pool combines agreement in log probability space and normalizes each fused row once.

TabICL uses the v1.1 checkpoint, 64 estimators, quantile normalization and temperature 0.9. It receives the unchanged first 152 columns of the same prepared table: two PCA views and the 24 fixed statistics. No extra language-model extraction or feature transform is introduced. Its raw class-aligned probabilities R contribute through the arithmetic mixture 0.95G + 0.05R, preserving their floating-point row mass. Punctuation correction with log penalty 2 follows this mixture, and the original hard Claude gate makes the final within-family decision. There is no second column balance. Each tabular model runs in a separate process so its GPU state is released before the next member starts.

On the reused partial prompt-disjoint development cache, the selected combination improved the prior-candidate proxy by 0.00417 mean balanced accuracy across three query contexts; the context differences were +0.00556, +0.00648 and +0.00046. The TabICL contribution against the otherwise identical geometric pool was +0.00201, positive in all three contexts. This configuration came from an exploratory strength-0.25 sensitivity analysis after the primary strength-0.5 TabICL comparisons were negative. The estimates are conditional on that selection. These caches omit several supervised production heads and the complete Claude specialist gate, so the comparisons are partial development evidence, not independent full-stack validation or a leaderboard forecast.

Credits: Lucas Boesen for the reference blend and trained readouts; Zbigniew Kirchner for the third LoRA and Claude gates; Obstacle Developer for centered stylegram; LIU ROUHAN for the weaker-balancing comparison; Vladislav Lavrentev for the original LoRA, tabular/masked-sequence representation and fusion study; saddamaditya for punctuation rules; kirilltym, pjmathematician and seyitkaangunes for upstream attention, encoder and public API-passage assets; Prior Labs for TabPFN 3.5; the TabICL authors for their classifier and checkpoint. Offline inputs retain their licenses and attribution. TabPFN weights use the Prior Labs Non-Commercial License. The TabICL v1.1 model card declares BSD-3-Clause; its pinned model-card evidence and the separate full package license are included in its offline asset. Other upstream licenses apply separately.


## Offline execution

The complete reference runs first, retaining its original gated predictions as a control. Its captured views and gate state stay temporary. After that process exits, train-only tabular preparation and PFN run in separate processes. Finalization checks gate replay against the actual control before writing the own predictions.


In [ ]:
from pathlib import Path
Path('/kaggle/temp/ntl-v6-pfn184-code').mkdir(parents=True, exist_ok=False)


### Centered stylegram evidence

In [ ]:
%%writefile /kaggle/temp/ntl-v6-pfn184-code/centered_stylegram_adapter.py
"""Soft log-evidence extension of adaptive centered stylegram.

Reference recipe: OBSTACLE DEVELOPER, Kaggle notebook139200100 v2,
code SHA256 b856e736d5cb47d8fa309ce2acd5ce0588a4f91478bdeae99c491f31b7c26a4c.
The upstream stack credits Lucas Boesen, Zbigniew Kirchner, Vladislav
Lavrentev, kirilltym, pjmathematician and seyitkaangunes. Source prose
declares Apache2.0; existing upstream licenses apply separately.

Own extension: interpolate clipped LOG evidence on eligible query clusters
instead of always replacing the plain component. No row normalization is
applied after mixing. This is an inference adapter, not validation evidence.
"""

import numpy as np
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.sparse import csr_matrix, diags
from scipy.spatial.distance import squareform
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

CLIP_MIN = 1e-9
MIN_GROUP = 2
MAX_GROUP = 18
CONTENT_THRESHOLD = 0.90


def center_sparse_groups(features, groups):
    """Subtract eligible group means without scaling or renormalizing."""
    matrix = csr_matrix(features)
    groups = np.asarray(groups)
    if groups.ndim != 1 or len(groups) != matrix.shape[0]:
        raise ValueError("One training/query group per feature row is required")
    _, inverse, counts = np.unique(groups, return_inverse=True, return_counts=True)
    eligible_groups = np.flatnonzero((counts >= MIN_GROUP) & (counts <= MAX_GROUP))
    if not len(eligible_groups):
        return matrix.copy()
    column_map = np.full(len(counts), -1, dtype=np.int32)
    column_map[eligible_groups] = np.arange(len(eligible_groups), dtype=np.int32)
    mapped_columns = column_map[inverse]
    rows = np.flatnonzero(mapped_columns >= 0)
    membership = csr_matrix(
        (np.ones(len(rows), dtype=matrix.dtype), (rows, mapped_columns[rows])),
        shape=(matrix.shape[0], len(eligible_groups)), dtype=matrix.dtype)
    means = diags((1.0 / counts[eligible_groups]).astype(matrix.dtype)) @ (
        membership.T @ matrix)
    return (matrix - membership @ means).tocsr()


def query_content_groups(query_texts):
    """Infer groups solely from the current query texts; no query prompt IDs."""
    texts = list(query_texts)
    if len(texts) < 2:
        return np.ones(len(texts), dtype=int)
    vectorizer = TfidfVectorizer(
        stop_words="english", sublinear_tf=True, min_df=1,
        max_features=100_000, dtype=np.float32)
    content = vectorizer.fit_transform(texts)
    distances = np.clip(1.0 - (content @ content.T).toarray(), 0.0, 1.0)
    np.fill_diagonal(distances, 0.0)
    upper = np.triu_indices(len(texts), 1)
    median_distance = max(float(np.median(distances[upper])), 1e-6)
    distances /= median_distance
    np.fill_diagonal(distances, 0.0)
    return fcluster(
        linkage(squareform(distances, checks=False), method="average"),
        CONTENT_THRESHOLD, criterion="distance")


def mix_stylegram_log_evidence(plain_probabilities, centered_probabilities,
                              eligible_rows, shrinkage):
    """Return exact add()-compatible evidence; endpoints have no extra math."""
    weight = float(shrinkage)
    if not np.isfinite(weight) or not 0.0 <= weight <= 1.0:
        raise ValueError("shrinkage must be finite and in [0, 1]")
    plain = np.asarray(plain_probabilities)
    eligible = np.asarray(eligible_rows, dtype=bool)
    if plain.ndim != 2 or eligible.shape != (plain.shape[0],):
        raise ValueError("Plain probabilities and eligible rows are misaligned")
    if not np.isfinite(plain).all() or (plain < 0).any() or (plain > 1).any():
        raise ValueError("Plain probabilities must be finite in [0, 1]")
    evidence = np.log(np.clip(plain, CLIP_MIN, 1))
    if weight == 0.0 or not eligible.any():
        return evidence
    centered = np.asarray(centered_probabilities)
    if centered.shape != (int(eligible.sum()), plain.shape[1]):
        raise ValueError("Centered probabilities must align with eligible rows")
    if not np.isfinite(centered).all() or (centered < 0).any() or (centered > 1).any():
        raise ValueError("Centered probabilities must be finite in [0, 1]")
    centered_log = np.log(np.clip(centered, CLIP_MIN, 1))
    if weight == 1.0:
        evidence[eligible] = centered_log
    else:
        evidence[eligible] = ((1.0 - weight) * evidence[eligible]
                              + weight * centered_log)
    return evidence


def centered_stylegram_log_evidence(train_features, query_features,
                                   train_labels, train_prompt_ids,
                                   query_texts, plain_probabilities,
                                   shrinkage, *, adapt_enabled=True):
    """Fit at most one centered SVC; return (log_evidence, small diagnostics).

    Matrices must be the EXISTING four-view float64 stylegram TF-IDF CSR
    matrices in unchanged row/column order, fitted on official TRAIN only.
    train_labels must use canonical integer class indices 0..17.
    query_texts must have exactly the same order as query_features/plain P.
    No query labels or true query prompt IDs are accepted by this API.
    """
    train_matrix = csr_matrix(train_features)
    query_matrix = csr_matrix(query_features)
    labels = np.asarray(train_labels)
    prompt_ids = np.asarray(train_prompt_ids)
    texts = list(query_texts)
    plain = np.asarray(plain_probabilities)
    if train_matrix.dtype != np.float64 or query_matrix.dtype != np.float64:
        raise ValueError("Original stylegram float64 feature dtype is required")
    if train_matrix.shape[1] != query_matrix.shape[1]:
        raise ValueError("Train/query feature columns differ")
    if labels.shape != (train_matrix.shape[0],) or prompt_ids.shape != labels.shape:
        raise ValueError("Training labels/prompt IDs are misaligned")
    if plain.shape != (query_matrix.shape[0], 18) or len(texts) != len(plain):
        raise ValueError("Query rows and canonical 18-class columns must align")
    if not np.array_equal(np.unique(labels), np.arange(18)):
        raise ValueError("Training must contain all canonical classes 0..17")
    # Validate lambda even when adaptation is disabled.
    weight = float(shrinkage)
    if not np.isfinite(weight) or not 0.0 <= weight <= 1.0:
        raise ValueError("shrinkage must be finite and in [0, 1]")
    eligible = np.zeros(len(texts), dtype=bool)
    diagnostics = {
        "method": "conditional_stylegram_soft_log_evidence",
        "shrinkage": weight, "query_rows": len(texts), "eligible_rows": 0,
        "centered_svc_fitted": False, "adapt_enabled": bool(adapt_enabled),
        "group_min": MIN_GROUP, "group_max": MAX_GROUP,
        "content_threshold": CONTENT_THRESHOLD,
    }
    if not adapt_enabled or weight == 0.0:
        return mix_stylegram_log_evidence(plain, None, eligible, weight), diagnostics

    # Same training geometry as the public hard-replacement recipe, including
    # uncentered training groups outside the eligible size range.
    centered_train = center_sparse_groups(train_matrix, prompt_ids)
    model = LinearSVC(C=0.5, random_state=42, max_iter=4000).fit(
        centered_train, labels)
    diagnostics["centered_svc_fitted"] = True
    if not np.array_equal(model.classes_, np.arange(18)):
        raise ValueError("Centered SVC class order differs from canonical indices")
    inferred_groups = query_content_groups(texts)
    sizes = np.bincount(inferred_groups)[inferred_groups]
    eligible = (sizes >= MIN_GROUP) & (sizes <= MAX_GROUP)
    diagnostics["eligible_rows"] = int(eligible.sum())
    centered_probabilities = None
    if eligible.any():
        centered_query = center_sparse_groups(
            query_matrix[eligible], inferred_groups[eligible])
        scores = model.decision_function(centered_query)
        exp_scores = np.exp(scores - scores.max(1, keepdims=True))
        centered_probabilities = np.zeros((int(eligible.sum()), 18))
        centered_probabilities[:, model.classes_] = (
            exp_scores / exp_scores.sum(1, keepdims=True))
    evidence = mix_stylegram_log_evidence(
        plain, centered_probabilities, eligible, weight)
    return evidence, diagnostics



### Fixed configuration


In [ ]:
%%writefile /kaggle/temp/ntl-v6-pfn184-code/runtime_config.json
{
  "reference_notebook": "zbigniewkirchner/ntl-v6",
  "reference_version": 2,
  "reference_scored_script_version_id": 359152926,
  "reference_source_sha256": "74c8d75a64cb4f64b4e82906e267fd098e230f539463ddb0ef10ea871d117331",
  "tabpfn": {
    "checkpoint_filename": "tabpfn-v3.5-20260909.safetensors",
    "checkpoint_sha256": "ece4d67eadfea42eb0e610df5189bea60cb7f31073d81e9c7a019b76eacf0be3",
    "checkpoint_bytes": 876027932,
    "package_version": "9.0.0",
    "n_estimators": 128
  },
  "offline_asset_directory_name": "ntl-tabular-offline-assets",
  "pca_components_per_view": 64,
  "pca_iterated_power": 3,
  "random_state": 42,
  "probability_transform": "none",
  "orthography_post_penalty": 2.0,
  "second_column_balance": false,
  "baseline_timeout_seconds": 13200,
  "prepare_timeout_seconds": 300,
  "model_timeout_seconds": 3600,
  "package_install_timeout_seconds": 300,
  "verified_runtime_imports": {
    "numpy": "2.1.3",
    "pandas": "2.3.3",
    "scipy": "1.16.3",
    "sklearn": "1.6.1",
    "torch": "2.11.0+cu128",
    "tabpfn": "9.0.0"
  },
  "feature_count": 184,
  "text_feature_arm": "masked_sequence",
  "masked_feature_module_sha256": "ce6313975b0f8eb8a3b7c8ce9cd0d1656d0e16bd1f7340d50dbc42ec1ac991b2",
  "text_benchmark_source_sha256": "601f5124682320a970f4be51873d70b035f5c11fe3569ce244b0bad21a4dd665",
  "masked_svd_components_per_view": 16,
  "rank_refinement_source_sha256": "b46fb37329d4075ee58a4d0b2920b86956885cd0832b2b5b85c2316554b2ff33",
  "rank_refinement_spec_sha256": "0f4fc0deab0d4cc1ef7146553079efbccfa24fe20afac14bbfaab2e21eb83fc1",
  "runtime_diagnostic_script_version_id": 359162368,
  "runtime_diagnostic_query_capacity_rows": 3,
  "larger_feature_capacity_reference": {
    "script_version_id": 359154729,
    "features": 216,
    "query_rows": 4096
  },
  "reference_scored_version": 1,
  "final_claude_gate": true,
  "baseline_balance_alpha": 0.25,
  "centered_stylegram_shrinkage": 1.0,
  "centered_stylegram_adapter_sha256": "4e1158f9d30d23f0cece8477f7330038edf9ee41e096aae78b5c6f006d955214",
  "parent_own_source_sha256": "b040fd59cb205a27e69b6a189a28c2c239e09a57b452ba1cccf7f2e88198c3a4",
  "fusion": {
    "kind": "normalized_geometric",
    "base_exponent": 0.8,
    "tabpfn_exponent": 0.2,
    "probability_floor": 1E-300,
    "input_probability_transform": "none"
  },
  "tabicl_mix": {
    "geometric_weight": 0.95,
    "tabicl_weight": 0.05,
    "probability_transform": "none",
    "row_renormalized": false
  },
  "tabicl": {
    "checkpoint_filename": "tabicl-classifier-v1.1-20250506.ckpt",
    "checkpoint_sha256": "bb42d1733b49fab3a7a9e42db648c8ff64d8d5705d93c039179add3bfb2db32b",
    "checkpoint_bytes": 108318621,
    "package_version": "2.2.0",
    "n_estimators": 64,
    "wheel_filename": "tabicl-2.2.0-py3-none-any.whl",
    "wheel_sha256": "c80b8ec2719be26bc0bf1dd758a2361f58019d8f7877038b785529a425789ce4"
  },
  "tabicl_timeout_seconds": 1800,
  "tabicl_offline_asset_directory_name": "ntl-tabicl-offline-assets"
}

### Exact ntl-v6 version 2 reference


In [ ]:
%%writefile /kaggle/temp/ntl-v6-pfn184-code/ntl_reference.py
# Name That LLM - v5: log-prob blend of
#   style-aware n-grams + LinearSVC | stylometric features + LightGBM |
#   Qwen3-8B-Base L7+L22 and Qwen2.5-7B L6+L20 (mean-pooled hidden states, layers concatenated) + LR |
#   public UltraBERT-380M 5-fold ensemble.
# CV 0.4927 (half-split held-out blend 0.4928) vs 0.450 for v4 (LB 0.436): paired +0.033 [+0.018, +0.048].
# Notebook settings: GPU T4 x2, internet off. Inputs: competition data; models qwen-lm/qwen-3 (transformers 8b-base),
# qwen-lm/qwen2.5 (transformers 7b); dataset pjmathematician/ultrabert-380m-ntl (fold weights, code = dmis-lab's).
# v7 = v6 + Qwen3-4B style-token-pooled head (L4+L7) + GLM-4-9B-0414 head (L8+L3, weights from a training-kernel output).
# v6 = v5 + two transductive steps (no labels used): test-set standardization of the LM-head features and
# Sinkhorn marginal balancing of the final posterior (alpha 0.5).
# v8 = v7 + capped adaptive prompt-centering for the 4 LM heads: test passages whose topic cluster (content-word
# TF-IDF, average linkage, cosine-distance cut ADAPT_THRESH) has ADAPT_MIN..ADAPT_MAX members use a prompt-centered
# twin of each head on cluster-centered features; everything else keeps the plain head. Never centers singletons
# (v1's failure) and never centers clusters > 18 (certainly mixed prompts). Simulated on held-out prompts (pooled
# 150-prompt test, v7 stack): +0.000 (1-2 passages/prompt) / +0.019 (1-5) / +0.034 (3-8) / +0.056 (all 18).
# v9 = v8 with the centered twins ADDED as extra blend evidence (plain heads kept for every passage) from cluster
# size 2 (as in the public 0.490 notebook). Pooled 150-prompt sim on the v7 stack, add>=2 vs plain:
# +0.024 (1-2 passages/prompt) / +0.042 (1-5) / +0.043 (3-8) / +0.052 (all 18); v8's replace>=4: 0 / +0.019 / +0.034 / +0.056.
# v10 = v9 with better test grouping and stronger centered evidence (pooled sim by session 4e): clusters from the
# average of the content-word TF-IDF cosine distance and the GLM-4-9B layer-24 cosine distance (mean-centered,
# L2-normalized) cut at 0.62 (at 0.65: same-prompt pair recall 0.89 vs 0.78 for TF-IDF alone, precision 0.97); centered
# twins with C x1.0 (was x0.4) and their evidence x1.5. Pooled-sim grid vs v9: 1-5 passages/prompt about +0.008,
# 1-2 about +0.003..+0.008, 3-8 / all-18 about +0.02; non-negative in every cell.
# v11 = v10 + a FROZEN DeBERTa-v3-large head (masked mean pool of layers 24+6, StandardScaler + LR C=0.005; CV 0.445
# alone, held-out +0.0061 on the v7 set, 18/20 splits; idea from Rudra's public fusion, which uses a frozen v3-base).
# Plain blend weights refit (L-BFGS, seed-0 OOF). The centered evidence is DECOUPLED from them: each twin gets an
# explicit weight TWIN_W (v10's 1.5 x v10 weight for the 4 LM twins, unchanged) plus a DeBERTa twin at 0.25.
# Pooled 150-prompt sim, 40 fresh draws, paired vs v10: 1-2 / 1-5 / 2-4 / 3-8 passages per prompt =
# +0.010 / +0.007 / +0.007 / +0.013 (SE 0.001-0.003; best of 6 variants, all of which beat v10 everywhere).
# v12 = v11 with better frozen encoders: DeBERTa-v3-large mean L23 + style-token pool L23 (0.450 alone) and
# ELECTRA-large discriminator mean L21 + style L24 (0.434). Held-out blend vs the v10 member set: +0.0157 (v11 set: +0.0061).
# v13 = v11 (same DeBERTa member and same twins) + frozen ELECTRA-large + frozen desklib AI-text detector (a DeBERTa-v3-large
# fine-tuned to detect AI text; mean L23 + style L23, 0.428 alone). Plain held-out +0.0088 vs the v11 set (20/20). In the
# pooled adapt sim v12 tied v11, while this set (v11 twins kept) beat v11 by +0.008/+0.000/+0.005/+0.003 (1-2/1-5/2-4/3-8).
# v13 public LB 0.532 (v8 0.499).
# v14 = v13 + a public LoRA classifier: Qwen3-8B-Base truncated to 24 layers, LoRA r16 on all linear layers, mean pool
# (first token masked) of layers 2/4/6/8/12/16/20/24, each with its own LayerNorm, concatenated -> linear head; trained on
# same-prompt triplets with plain + centered heads (kernel output lightningv08/ntl-lora24-fivefold-training, 'final' fit;
# its 5-fold OOF plain head = 0.4816 balanced accuracy). Here: inference only, plain head, as one more blend member.
# v15 = v14 + the LoRA's CENTERED head as a twin on clustered test rows (cluster-centered LoRA features -> centered
# readout, weight 0.15) + a learned token-attention head on frozen Qwen3.5-9B layer-18 token states (public recipe
# kirilltym/0-525-learned-token-attention, trained by kernel lucasboesen/name-that-llm-train-tokattn; OOF 0.394 alone).
# v16 = v15 + our own run of the LoRA24 recipe (seed 7, our folds; kernel lucasboesen/name-that-llm-train-lora24-s7; OOF
# 0.4804, argmax agreement with the public LoRA 0.69) + the same token-attention head on frozen DeBERTa-v3-large L23
# (kernel lucasboesen/name-that-llm-train-tokattn2; OOF 0.411). Held-out +0.0060 vs the v15 set (20/20 splits).
ADAPT, ADAPT_THRESH, ADAPT_MIN, ADAPT_MAX, ADAPT_C_MULT = True, 0.85, 2, 18, 1.0
TWIN_W = {'q3_8b_cat7+22': 1.5 * 0.219, 'q25_7b_cat6+20': 1.5 * 0.141, 'style4b': 1.5 * 0.083, 'glm9b': 1.5 * 0.110,
          'debv3l': 0.25}   # centered-evidence weight per twin (add mode); the first four = v10 exactly
CLUSTER_MODE, CLUSTER_THRESH_MIX, CLUSTER_LAYER = 'mixglm', 0.60, 24   # 'tfidf' = v9 clustering; 0.60: false-positive
#   pair rate at m=150 prompts sits between th0.58 (0.012, R 0.72) and th0.62 (0.059, R 0.85); tfidf 0.85: 0.053, R 0.80
ADAPT_MODE = 'add'     # 'add': plain + centered evidence on cluster rows; 'replace': centered only (v8)
SMOKE = False          # local code-path test: tiny models, 200 train rows, CPU ok
BALANCE_ALPHA = 0.5
TRANS_STD_MIN = 2      # min test rows for test-set standardization (2 so the 3-row interactive run exercises it)
W = {'stylegram': 1.516, 'q3_8b_cat7+22': 0.0, 'q25_7b_cat6+20': 0.0, 'style_lgb': 0.063, 'ultrabert': 0.095,
     'style4b': 0.0, 'glm9b': 0.0, 'debv3l': 0.05, 'electl': 0.032, 'desk': 0.132, 'lora': 0.177, 'tokattn': 0.153,
     'lora2': 0.113, 'lora3': 0.177, 'tokattn_deb': 0.185}
# L-BFGS on seed-0 OOF; held-out +0.0060 vs the v15 set (20/20 splits). Pooled adapt sim vs v15 (60 fresh draws): a TIE
# (+0.000 / +0.001 / +0.002 / -0.001), so v16 is submitted as an EXPERIMENT (does a plain-only gain transfer to the LB?).
LORA = dict(enabled=True, layers=[2, 4, 6, 8, 12, 16, 20, 24], max_len=447, bs=8)
LORAS = [  # (key, checkpoint marker glob, norms file, anchor-features file): all share the Qwen3-8B-Base 24-layer trunk
    ('lora', '**/lora24/final/COMPLETE.json', 'heads.safetensors', 'anchor_features.npz'),   # public lightningv08 run
    ('lora2', '**/lora_out/job_final/COMPLETE.json', 'norms.safetensors', 'valid_features.npz'),   # our seed-7 run
    ('lora3', '**/ntl-qwen3-8b-lora-claude-head/**/final/COMPLETE.json', 'heads.safetensors', 'anchor_features.npz'),   # zbigniewkirchner: + Claude head
]
TWIN_W_LORA = {'lora': 0.15, 'lora3': 0.15}   # centered-readout evidence on clustered rows (add mode)
TOKATTN = dict(enabled=True, bs=8)
TOKATTN_HEADS = [  # (key, head file from a training-kernel output, backbone)
    ('tokattn', 'tokattn_final.pt', 'qwen3_5'),            # Qwen3.5-9B L18 (decoder, fp16)
    ('tokattn_deb', 'tokattn_debl23_final.pt', 'deberta'),   # DeBERTa-v3-large L23 (encoder, fp32)
]
ENCODERS = [  # (key, Kaggle input dir name holding fold0..fold4, config model_type, trust_remote_code, local fallbacks)
    ('ultrabert', 'ultrabert-380m-ntl', 'ultrabert', True,
     ['external/ultrabert-380m-ntl', '../../external/ultrabert-380m-ntl']),
]
LLM_HEADS = [  # (key, model_type, hidden_size, layers (concatenated), C, smoke HF id, pooling)
    ('q3_8b_cat7+22', 'qwen3', 4096, [7, 22], 0.0025, 'Qwen/Qwen3-0.6B-Base', 'mean'),
    ('q25_7b_cat6+20', 'qwen2', 3584, [6, 20], 0.0025, 'Qwen/Qwen2.5-0.5B', 'mean'),
    ('style4b', 'qwen3', 2560, [4, 7], 0.0025, 'Qwen/Qwen3-0.6B-Base', 'style'),   # Qwen3-4B-Base, style-token pooling
    ('glm9b', 'glm4', 4096, [8, 3], 0.0025, 'Qwen/Qwen2.5-0.5B', 'mean'),         # GLM-4-9B-0414 truncated to 25 layers
]
FROZEN_ENC = [  # (key, input dir name, [(layer, pool)] concatenated, C, smoke HF id): bidirectional encoders, fp32
    ('debv3l', 'deberta-v3-large', [(24, 'mean'), (6, 'mean')], 0.005, 'microsoft/deberta-v3-xsmall'),   # = v11 member
    ('electl', 'electra-large-discriminator', [(21, 'mean'), (24, 'style')], 0.005, 'google/electra-small-discriminator'),
    ('desk', 'desklib-detector', [(23, 'mean'), (23, 'style')], 0.005, 'microsoft/deberta-v3-xsmall'),
]   # weights: kernel outputs name-that-llm-extract-encoders2, -electra-large-weights, -desklib-weights (found by dir name:
#     desklib and DeBERTa-v3-large share model_type/hidden size)

import gc
import glob
import json
import os
import pathlib
import re
import time

import numpy as np
import pandas as pd
import torch
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.sparse import hstack
from scipy.spatial.distance import squareform
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from transformers import AutoConfig, AutoModel, AutoModelForSequenceClassification, AutoTokenizer

T0 = time.time()
LABELS = sorted([
    'gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra',
    'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash',
    'claude-fable-5', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5',
    'qwen3.8-max', 'glm-5.3', 'glm-5.3-flash', 'kimi-k3'])
K = len(LABELS)
DATA_DIR = next(p for p in [pathlib.Path('/kaggle/input/competitions/name-that-llm'),
                            pathlib.Path('/kaggle/input/name-that-llm'),
                            pathlib.Path('data')] if (p / 'train.csv').exists())
train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
if SMOKE:
    train = train.sample(200, random_state=0).reset_index(drop=True)
test['text'] = test['text'].fillna('').astype(str)
y = train['model'].map({l: i for i, l in enumerate(LABELS)}).values
TR, TE = train['text'].tolist(), test['text'].tolist()
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
NGPU = torch.cuda.device_count()
print('train', train.shape, 'test', test.shape, 'gpus', NGPU, flush=True)
LOGIT = np.zeros((len(test), K))


def add(key, prob):
    LOGIT[:] += W[key] * np.log(np.clip(prob, 1e-9, 1))
    print(f'{key} done ({time.time() - T0:.0f}s)', flush=True)


def topic_clusters(texts, thresh):
    """Average-linkage clusters on cosine distance of content-word TF-IDF (stop words removed)."""
    if len(texts) < 2:
        return np.ones(len(texts), dtype=int)
    X = TfidfVectorizer(stop_words='english', sublinear_tf=True).fit_transform(texts)
    D = np.clip(1 - (X @ X.T).toarray(), 0, None)
    np.fill_diagonal(D, 0)
    return fcluster(linkage(squareform(D, checks=False), method='average'), thresh, criterion='distance')


def group_center(X, groups):
    X = np.asarray(X, dtype=np.float32).copy()
    for g in np.unique(groups):
        i = groups == g
        X[i] -= X[i].mean(0)
    return X


def mixed_clusters(texts, E, thresh):
    """Average-linkage clusters on the mean of the TF-IDF cosine distance and the cosine distance of the
    mean-centered, L2-normalized embeddings E (GLM-4-9B layer 24)."""
    if len(texts) < 2:
        return np.ones(len(texts), dtype=int)
    X = TfidfVectorizer(stop_words='english', sublinear_tf=True).fit_transform(texts)
    Dt = np.clip(1 - (X @ X.T).toarray(), 0, None)
    Z = np.asarray(E, dtype=np.float64)
    Z = Z - Z.mean(0)
    Z /= np.linalg.norm(Z, axis=1, keepdims=True) + 1e-12
    D = (Dt + np.clip(1 - Z @ Z.T, 0, None)) / 2
    np.fill_diagonal(D, 0)
    return fcluster(linkage(squareform(D, checks=False), method='average'), thresh, criterion='distance')


def compute_groups(E_te=None, E_selftest=None):
    """Adaptive-centering targets: test passages in clusters of size ADAPT_MIN..ADAPT_MAX (+ tiny-test self-test)."""
    global G_TE, ADAPT_ROWS, SELFTEST
    use_mix = CLUSTER_MODE == 'mixglm' and E_te is not None
    G_TE = (mixed_clusters(TE, E_te, CLUSTER_THRESH_MIX) if use_mix else topic_clusters(TE, ADAPT_THRESH)) \
        if ADAPT else np.ones(len(TE), dtype=int)
    sz = np.bincount(G_TE)[G_TE]
    ADAPT_ROWS = ADAPT & (sz >= ADAPT_MIN) & (sz <= ADAPT_MAX)
    print('clustering', 'mixglm' if use_mix else 'tfidf', '| test cluster sizes (passages): 1:', int((sz == 1).sum()),
          ' 2-3:', int(((sz >= 2) & (sz <= 3)).sum()), ' 4-18:', int(((sz >= 4) & (sz <= 18)).sum()),
          ' >18:', int((sz > 18).sum()), ' | adapt rows:', int(ADAPT_ROWS.sum()), ADAPT_MODE, flush=True)
    SELFTEST = None
    if ADAPT and len(TE) < 50:   # interactive 3-row test: exercise the adapt branch on train rows (output discarded)
        st = np.where(train['prompt_id'].isin(train['prompt_id'].drop_duplicates().iloc[:3]).values)[0]
        txt = [TR[i] for i in st]
        g = mixed_clusters(txt, E_selftest[st], CLUSTER_THRESH_MIX) if (use_mix and E_selftest is not None) \
            else topic_clusters(txt, ADAPT_THRESH)
        s = np.bincount(g)[g]
        keep = (s >= ADAPT_MIN) & (s <= ADAPT_MAX)
        SELFTEST = (st[keep], g[keep])
        print('adapt self-test rows:', int(keep.sum()), 'of', len(st), flush=True)
        if use_mix and E_selftest is not None:   # clustering QA on a train-built pseudo-test (1-5 passages/prompt)
            rng = np.random.RandomState(0)
            pid = train['prompt_id'].values
            qa = np.concatenate([rng.choice(np.where(pid == p)[0], min(rng.randint(1, 6), int((pid == p).sum())),
                                            replace=False) for p in np.unique(pid)])
            gq = mixed_clusters([TR[i] for i in qa], E_selftest[qa], CLUSTER_THRESH_MIX)
            sq = np.bincount(gq)[gq]
            same_c = gq[:, None] == gq[None, :]
            same_p = pid[qa][:, None] == pid[qa][None, :]
            off = ~np.eye(len(qa), dtype=bool)
            tp = (same_c & same_p & off).sum()
            print(f'clustering QA on {len(qa)} train passages (1-5/prompt): pair precision '
                  f'{tp / max((same_c & off).sum(), 1):.3f} recall {tp / max((same_p & off).sum(), 1):.3f} | sizes 1:',
                  int((sq == 1).sum()), ' 2-3:', int(((sq >= 2) & (sq <= 3)).sum()), ' 4-18:',
                  int(((sq >= 4) & (sq <= 18)).sum()), ' >18:', int((sq > 18).sum()), flush=True)


G_TE, ADAPT_ROWS, SELFTEST = None, None, None   # set by compute_groups() after the GLM pass (section 3)


# ------------------------------------------------ 1) style-aware n-grams + LinearSVC
FUNCTION_WORDS = set("a an the and or but if while although because as so than that which who whom whose this these those it its they them their theirs he him his she her hers we us our ours you your yours i me my mine in on at to from with without for of by over under between through into about is are was were be been being have has had do does did can could will would shall should may might must not no nor all any some each every both either neither other another such only also just very too more most less least here there when where why how what whether then now still yet even much many few little".split())
SPEC = [('char', 3, 5, False, 250000, 1.0), ('word', 1, 3, False, 180000, 0.7),
        ('shape', 2, 6, False, 60000, 0.5), ('function', 1, 3, False, 50000, 0.5)]


def shape_text(text):
    return re.sub(r"[^\W\d_]+", lambda m: "X" if m[0].isupper() else "x", text)


def function_text(text):
    return " ".join(t.lower() if t.lower() in FUNCTION_WORDS or not t.isalnum() else "CONTENT"
                    for t in re.findall(r"\w+|[^\w\s]", text))


A, B = [], []
for analyzer, lo, hi, lower, maxf, w in SPEC:
    tf = shape_text if analyzer == 'shape' else function_text if analyzer == 'function' else str
    actual = 'char' if analyzer == 'shape' else 'word' if analyzer == 'function' else analyzer
    opts = dict(analyzer=actual, ngram_range=(lo, hi), lowercase=lower, sublinear_tf=True, min_df=2,
                max_features=maxf, dtype=np.float64)
    if actual == 'word':
        opts['token_pattern'] = r"(?u)\b\w+\b|[^\w\s]"
    v = TfidfVectorizer(**opts)
    A.append(v.fit_transform([tf(t) for t in TR]) * w)
    B.append(v.transform([tf(t) for t in TE]) * w)
svc = LinearSVC(C=0.5, random_state=42, max_iter=4000).fit(hstack(A, format='csr'), y)
s = svc.decision_function(hstack(B, format='csr'))
s = np.exp(s - s.max(1, keepdims=True))
p = np.zeros((len(TE), K))
p[:, svc.classes_] = s / s.sum(1, keepdims=True)
add('stylegram', p)
del A, B, svc

# ------------------------------------------------ 1b) within-Claude SVC on original Claude rows + free public Claude API passages
# (dataset seyitkaangunes/ntl-extra-api-passages). Used only to choose WHICH Claude after the final blend says "Claude".
# Local GroupKFold CV on the stylegram model: 0.3845 -> 0.3967 (Claude recall 0.393 -> 0.438, other classes unchanged).
CLAUDE_GATE_LAM = 1.5
CLAUDE_LORA_LAM = 1.0
LC_TE = None
_api_hits = glob.glob('/kaggle/input/**/ntl_extra_passages_claude.csv', recursive=True)
CLAUDE_IDX = np.array([i for i, l in enumerate(LABELS) if l.startswith('claude')])
DC_TE = None
if _api_hits:
    _api = pd.read_csv(_api_hits[0])
    _m = np.isin(y, CLAUDE_IDX)
    _ft = [t for t, k in zip(TR, _m) if k] + _api['text'].astype(str).tolist()
    _fy = np.concatenate([y[_m], _api['model'].map({l: i for i, l in enumerate(LABELS)}).values])
    A, B = [], []
    for analyzer, lo, hi, lower, maxf, w in SPEC:
        tf = shape_text if analyzer == 'shape' else function_text if analyzer == 'function' else str
        actual = 'char' if analyzer == 'shape' else 'word' if analyzer == 'function' else analyzer
        opts = dict(analyzer=actual, ngram_range=(lo, hi), lowercase=lower, sublinear_tf=True, min_df=2,
                    max_features=maxf, dtype=np.float64)
        if actual == 'word':
            opts['token_pattern'] = r"(?u)\b\w+\b|[^\w\s]"
        v = TfidfVectorizer(**opts)
        A.append(v.fit_transform([tf(t) for t in _ft]) * w)
        B.append(v.transform([tf(t) for t in TE]) * w)
    _svc = LinearSVC(C=0.5, random_state=42, max_iter=4000).fit(hstack(A, format='csr'), _fy)
    assert list(_svc.classes_) == list(CLAUDE_IDX)
    DC_TE = _svc.decision_function(hstack(B, format='csr'))
    print('claude gate model trained on', len(_ft), 'rows', flush=True)
    del A, B, _svc, _api
else:
    print('WARNING: Claude API passages not attached, gate disabled', flush=True)


# ------------------------------------------------ 2) stylometric features + LightGBM
PUNCT = list(',.;:!?()[]"\'-') + ['—', '–', '‘', '’', '“', '”', '…', '*', '/', '%']
FUNC = sorted(ENGLISH_STOP_WORDS)
SENT = re.compile(r'(?<=[.!?])\s+')
WORD = re.compile(r"[A-Za-z’']+")


def stylometric_features(t):
    words = WORD.findall(t)
    nw, nc = max(len(words), 1), max(len(t), 1)
    sents = [x for x in SENT.split(t.strip()) if x]
    sl = np.array([len(WORD.findall(x)) for x in sents]) if sents else np.array([0])
    wl = np.array([len(x) for x in words]) if words else np.array([0])
    lw = [x.lower() for x in words]
    f = [np.log1p(nw), len(sents), sl.mean(), sl.std(), sl.max(), sl.min(), wl.mean(), wl.std(),
         (wl >= 9).mean(), len(set(lw)) / nw, sum(c.isupper() for c in t) / nc, sum(c.isdigit() for c in t) / nc,
         sum(ord(c) > 127 for c in t) / nc, t.count(', ') / nw, t.count(' and ') / nw, t.count(' but ') / nw,
         sum(1 for x in sents if x[:1].isupper() and x.split()[0].lower() in ('this', 'these', 'it', 'they')) / max(len(sents), 1),
         sum(1 for x in words if x.endswith('ly')) / nw, sum(1 for x in words if x.endswith('ing')) / nw,
         sum(1 for x in words if x.endswith('tion')) / nw]
    f += [t.count(q) / nw for q in PUNCT]
    cnt = {}
    for x in lw:
        cnt[x] = cnt.get(x, 0) + 1
    f += [cnt.get(x, 0) / nw for x in FUNC]
    return f


import lightgbm as lgb
lg = lgb.LGBMClassifier(n_estimators=400, learning_rate=0.03, num_leaves=15, min_child_samples=10, subsample=0.8,
                        subsample_freq=1, colsample_bytree=0.5, reg_lambda=1.0, random_state=0, verbose=-1, n_jobs=4)
lg.fit(np.array([stylometric_features(t) for t in TR], dtype=np.float32), y)
p = np.zeros((len(TE), K))
p[:, lg.classes_] = lg.predict_proba(np.array([stylometric_features(t) for t in TE], dtype=np.float32))
add('style_lgb', p)


# ------------------------------------------------ 3) frozen LLM heads (truncated to layer+1, fp16 on GPU)
def find_model_dir(model_type, hidden):
    for cfg in sorted(glob.glob('/kaggle/input/**/config.json', recursive=True)):
        try:
            c = json.load(open(cfg))
        except Exception:
            continue
        if c.get('model_type') == model_type and c.get('hidden_size') == hidden:
            return os.path.dirname(cfg)
    return None


def load_truncated(path, keep, dtype):
    cfg = AutoConfig.from_pretrained(path)
    cfg.num_hidden_layers = keep
    if getattr(cfg, 'layer_types', None):
        cfg.layer_types = cfg.layer_types[:keep]
    kw = dict(config=cfg, dtype=dtype)
    if DEV == 'cuda':
        kw['device_map'] = 'auto'
    return AutoModel.from_pretrained(path, **kw).eval()


def style_mask(tok):
    """Vocab mask of 'style' tokens: decoded piece is an English stop word or pure punctuation."""
    func = frozenset(ENGLISH_STOP_WORDS)
    pieces = [tok.decode([i]) for i in range(len(tok))]
    return torch.tensor([(p.strip().lower() in func) or bool(re.fullmatch(r'\s*[^\w\s]+\s*', p)) for p in pieces])


@torch.no_grad()
def embed_layers(model, tok, texts, layers, bs=8, max_len=512, pool='mean', smask=None):
    """Pooled hidden_states[L] for each L in layers, concatenated -> (n, len(layers) * hidden).
    pool='mean': all tokens; pool='style': only tokens in smask (function words / punctuation)."""
    H = model.config.hidden_size
    out = np.zeros((len(texts), len(layers) * H), dtype=np.float32)
    order = np.argsort([len(t) for t in texts])
    dev = next(model.parameters()).device
    bad = 0
    for b0 in range(0, len(texts), bs):
        idx = order[b0:b0 + bs]
        batch = [(tok.bos_token + texts[i]) if tok.bos_token else texts[i] for i in idx]
        enc = tok(batch, return_tensors='pt', padding=True, truncation=True, max_length=max_len,
                  add_special_tokens=False).to(dev)
        hs = model(**enc, output_hidden_states=True).hidden_states
        if not all(torch.isfinite(hs[L]).all() for L in layers):   # fp16 overflow guard: switch to bf16, redo
            bad += 1
            model.to(torch.bfloat16)
            hs = model(**enc, output_hidden_states=True).hidden_states
        for li, L in enumerate(layers):
            h = hs[L]
            m = enc['attention_mask'].float()
            if pool == 'style':
                m = m * smask.to(m.device)[enc['input_ids']].float()
            m = m.unsqueeze(-1).to(h.device)
            out[idx, li * H:(li + 1) * H] = ((h.float() * m).sum(1) / m.sum(1).clamp(min=1)).cpu().numpy()
        del hs, enc
    if bad:
        print('  non-finite fp16 batches redone in bf16:', bad, flush=True)
    return np.nan_to_num(out)


def extract(key, mt, hid, lays, smoke_id, pool):
    """Pooled hidden states of layers `lays` (concatenated) for train and test, model truncated to max(lays)+1."""
    path = smoke_id if SMOKE else find_model_dir(mt, hid)
    print(key, path, lays, pool, flush=True)
    assert path is not None, f'model for {key} ({mt}, hidden {hid}) not found under /kaggle/input'
    tok = AutoTokenizer.from_pretrained(path)
    tok.padding_side = 'right'
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    smask = style_mask(tok) if pool == 'style' else None
    dtype = torch.float16 if DEV == 'cuda' else torch.float32
    model = load_truncated(path, max(lays) + 1, dtype)
    if SMOKE:   # truncation must not change hidden_states[L]
        full = AutoModel.from_pretrained(path, dtype=dtype).eval()
        e1 = embed_layers(model, tok, TE, lays, pool=pool, smask=smask)
        e2 = embed_layers(full, tok, TE, lays, pool=pool, smask=smask)
        print('  truncation max abs diff', float(np.abs(e1 - e2).max()), flush=True)
        assert np.allclose(e1, e2, atol=1e-4)
        del full
    Etr = embed_layers(model, tok, TR, lays, pool=pool, smask=smask)
    Ete = embed_layers(model, tok, TE, lays, pool=pool, smask=smask)
    del model, tok
    gc.collect()
    if DEV == 'cuda':
        torch.cuda.empty_cache()
    return Etr, Ete


def smoke_layers(layers):
    return [min(L, 3 + i) for i, L in enumerate(layers)] if SMOKE else list(layers)


# GLM pre-pass: the GLM head's layers plus the clustering layer in one forward pass; then build the test groups.
CACHE = {}
_glm = next((h for h in LLM_HEADS if h[1] == 'glm4'), None)
if _glm is not None and CLUSTER_MODE == 'mixglm' and ADAPT:
    _gl = smoke_layers(_glm[3])
    _cl = 5 if SMOKE else CLUSTER_LAYER
    _Etr, _Ete = extract(_glm[0], _glm[1], _glm[2], _gl + [_cl], _glm[5], _glm[6])
    _H = _Etr.shape[1] // (len(_gl) + 1)
    CACHE[_glm[0]] = (_Etr[:, :len(_gl) * _H], _Ete[:, :len(_gl) * _H])
    compute_groups(_Ete[:, len(_gl) * _H:], _Etr[:, len(_gl) * _H:])
    del _Etr, _Ete
else:
    compute_groups()

def fit_head(key, Etr, Ete, C):
    """StandardScaler + LR head on pooled features; optional prompt-centered twin on clustered test rows."""
    sc = StandardScaler().fit(Etr)
    clf = LogisticRegression(C=C, max_iter=5000).fit(sc.transform(Etr), y)
    # transductive standardization: test features scaled with TEST mean/std (labels never used); CV: never
    # negative, +0.002..+0.01 depending on test size. Train stats fallback for tiny tests (3-row interactive file).
    Xte = (Ete - Ete.mean(0)) / (Ete.std(0) + 1e-6) if len(Ete) >= TRANS_STD_MIN else sc.transform(Ete)
    p = np.zeros((len(TE), K))
    p[:, clf.classes_] = clf.predict_proba(Xte)
    if ADAPT and key in TWIN_W:   # prompt-centered twin, used only on test passages in clusters of ADAPT_MIN..ADAPT_MAX
        Xc = group_center(Etr, train['prompt_id'].values)
        csc = StandardScaler().fit(Xc)
        cclf = LogisticRegression(C=ADAPT_C_MULT * C, max_iter=5000).fit(csc.transform(Xc), y)
        if ADAPT_ROWS.any():
            pc = np.zeros((int(ADAPT_ROWS.sum()), K))
            pc[:, cclf.classes_] = cclf.predict_proba(csc.transform(group_center(Ete[ADAPT_ROWS], G_TE[ADAPT_ROWS])))
            if ADAPT_MODE == 'replace':
                p[ADAPT_ROWS] = pc
            else:   # add the centered head's evidence on top of the plain head (applied after add(key, p))
                LOGIT[ADAPT_ROWS] += TWIN_W[key] * np.log(np.clip(pc, 1e-9, 1))
        if SELFTEST is not None and len(SELFTEST[0]):
            ps = cclf.predict_proba(csc.transform(group_center(Etr[SELFTEST[0]], SELFTEST[1])))
            print(f'  adapt self-test ({len(SELFTEST[0])} train rows, seen in training): '
                  f'acc {np.mean(cclf.classes_[ps.argmax(1)] == y[SELFTEST[0]]):.2f}', flush=True)
    add(key, p)


for key, mt, hid, layers, C, smoke_id, pool in LLM_HEADS:
    if key in CACHE:
        Etr, Ete = CACHE.pop(key)
    else:
        Etr, Ete = extract(key, mt, hid, smoke_layers(layers), smoke_id, pool)
    fit_head(key, Etr, Ete, C)


# ------------------------------------------------ 3b) frozen bidirectional encoders (fp32, masked mean pool incl. [CLS]/[SEP])
@torch.no_grad()
def embed_encoder(model, tok, texts, parts, smask, bs=16, max_len=512):
    """parts = [(layer, 'mean'|'style')]: masked mean of hidden_states[layer] over all tokens (incl. [CLS]/[SEP]) or
    over style tokens only (stop words / punctuation), concatenated -> (n, len(parts) * hidden)."""
    H = model.config.hidden_size
    out = np.zeros((len(texts), len(parts) * H), dtype=np.float32)
    order = np.argsort([len(t) for t in texts])
    dev = next(model.parameters()).device
    smask = smask.to(dev)
    for b0 in range(0, len(texts), bs):
        idx = order[b0:b0 + bs]
        enc = tok([texts[i] for i in idx], padding=True, truncation=True, max_length=max_len, return_tensors='pt').to(dev)
        hs = model(**enc, output_hidden_states=True).hidden_states
        m = enc['attention_mask'].unsqueeze(-1).float()
        ms = m * smask[enc['input_ids']].unsqueeze(-1).float()
        for li, (L, pool) in enumerate(parts):
            w = ms if pool == 'style' else m
            out[idx, li * H:(li + 1) * H] = ((hs[L].float() * w).sum(1) / w.sum(1).clamp(min=1)).cpu().numpy()
        del hs, enc
    return np.nan_to_num(out)


def find_dir(name):
    hits = sorted(glob.glob(f'/kaggle/input/**/{name}/config.json', recursive=True))
    return os.path.dirname(hits[0]) if hits else None


for key, dname, parts, C, smoke_id in FROZEN_ENC:
    path = smoke_id if SMOKE else find_dir(dname)
    print(key, path, parts, flush=True)
    assert path is not None, f'encoder for {key} (dir {dname}) not found under /kaggle/input'
    etok = AutoTokenizer.from_pretrained(path)
    emodel = AutoModel.from_pretrained(path, dtype=torch.float32).to(DEV).eval()
    if SMOKE:
        parts = [(min(L, 3 + i), pool) for i, (L, pool) in enumerate(parts)]
    sm = style_mask(etok)
    Etr, Ete = embed_encoder(emodel, etok, TR, parts, sm), embed_encoder(emodel, etok, TE, parts, sm)
    del emodel, etok
    gc.collect()
    if DEV == 'cuda':
        torch.cuda.empty_cache()
    fit_head(key, Etr, Ete, C)

# ------------------------------------------------ 3c) public LoRA classifier (inference only, plain head)
@torch.no_grad()
def run_loras(texts, parity_texts):
    """Pooled LoRA features of every checkpoint in LORAS (one Qwen3-8B-Base trunk, adapters swapped). For each: plain
    readout -> blend member W[key]; centered readout on cluster-centered features -> twin evidence TWIN_W_LORA[key]."""
    import importlib.util as _iu
    _fs = _iu.find_spec   # Kaggle ships torchao 0.10, which peft rejects; hide it (unused)
    _iu.find_spec = lambda name, *a, **k: None if name.split('.')[0] == 'torchao' else _fs(name, *a, **k)
    from peft import PeftModel
    import peft.import_utils as _pi
    _pi.is_torchao_available = lambda *a, **k: False
    from safetensors.torch import load_file
    base_dir = find_model_dir('qwen3', 4096)
    cfg = AutoConfig.from_pretrained(base_dir)
    cfg.num_hidden_layers, cfg.use_cache = 24, False
    if getattr(cfg, 'layer_types', None) is not None:
        cfg.layer_types = cfg.layer_types[:24]
    trunk = AutoModel.from_pretrained(base_dir, config=cfg, dtype=torch.float16, device_map='auto').eval()
    L = LORA['layers']
    for key, marker, norms_file, anchors_file in LORAS:
        hits = sorted(glob.glob('/kaggle/input/' + marker, recursive=True))
        assert hits, f'LoRA checkpoint for {key} not attached ({marker})'
        ck = pathlib.Path(os.path.dirname(hits[0]))
        print(key, ck, flush=True)
        ltok = AutoTokenizer.from_pretrained(ck / 'tokenizer')
        ltok.padding_side = 'right'
        if ltok.pad_token is None:
            ltok.pad_token = ltok.eos_token
        lm = PeftModel.from_pretrained(trunk, ck / 'adapter').eval()
        heads = load_file(str(ck / norms_file))
        norms = torch.nn.ModuleList([torch.nn.LayerNorm(4096) for _ in L])
        norms.load_state_dict({k[len('norms.'):]: v for k, v in heads.items() if k.startswith('norms.')})
        ro = np.load(ck / 'readouts.npz')
        assert list(ro['classes']) == LABELS and list(ro['pool_layers']) == L
        dev = lm.get_input_embeddings().weight.device
        norms = norms.to(dev).float()

        def embed(tl, bs):
            Z = np.zeros((len(tl), 4096 * len(L)), dtype=np.float32)
            order = np.argsort([len(t) for t in tl], kind='stable')
            for b0 in range(0, len(order), bs):
                idx = order[b0:b0 + bs]
                batch = ltok.pad({'input_ids': [tl[i] for i in idx]}, return_tensors='pt').to(dev)
                with torch.autocast('cuda', dtype=torch.float16):
                    hs = lm(**batch, output_hidden_states=True, use_cache=False).hidden_states
                m = batch['attention_mask'].unsqueeze(-1).float()
                m[:, 0] = 0
                Z[idx] = torch.cat([nm((hs[l].float() * m.to(hs[l].device)).sum(1).to(dev) / m.sum(1).to(dev))
                                    for l, nm in zip(L, norms)], -1).cpu().numpy()
                del hs, batch
            return Z

        def tokens_of(tx):
            return [ids + [ltok.eos_token_id] for ids in
                    ltok(list(tx), add_special_tokens=False, truncation=True, max_length=LORA['max_len'])['input_ids']]

        if (ck / anchors_file).exists():   # the checkpoint saved its features for train rows 0..2
            a = np.load(ck / anchors_file)['features'].astype(np.float32)
            Za = embed(tokens_of(parity_texts), 3)
            rel = float(np.abs(Za - a).mean() / (np.abs(a).mean() + 1e-9))
            print(f'  {key} parity vs saved anchor features: max abs {float(np.abs(Za - a).max()):.4f}, mean rel {rel:.5f}',
                  flush=True)
            assert rel < 0.02, f'{key}: features do not match the checkpoint anchors'
        Z = embed(tokens_of(texts), LORA['bs'])
        lg = Z @ ro['plain_coef'].T + ro['plain_intercept']
        lg -= lg.max(1, keepdims=True)
        p = np.exp(lg)
        add(key, p / p.sum(1, keepdims=True))
        if 'claude_coef' in ro.files:   # within-Claude head (orig Claude rows + public Claude API passages)
            global LC_TE
            lc5 = Z @ ro['claude_coef'].T + ro['claude_intercept']
            lc5 -= lc5.max(1, keepdims=True)
            LC_TE = lc5 - np.log(np.exp(lc5).sum(1, keepdims=True))
        if ADAPT and key in TWIN_W_LORA and ADAPT_ROWS.any():
            Zc = group_center(Z[ADAPT_ROWS], G_TE[ADAPT_ROWS])
            lc = Zc @ ro['center_coef'].T + ro['center_intercept']
            lc -= lc.max(1, keepdims=True)
            LOGIT[ADAPT_ROWS] += TWIN_W_LORA[key] * (lc - np.log(np.exp(lc).sum(1, keepdims=True)))
            print(f'  {key} centered twin on {int(ADAPT_ROWS.sum())} clustered rows', flush=True)
        trunk = lm.unload()   # drop this adapter, keep the trunk for the next checkpoint
        del lm
        gc.collect()
        torch.cuda.empty_cache()
    del trunk
    gc.collect()
    torch.cuda.empty_cache()


if LORA['enabled'] and not SMOKE:
    run_loras(TE, TR[:3])


# ------------------------------------------------ 3d) learned token-attention head on frozen Qwen3.5-9B layer-18 states
class TokAttn(torch.nn.Module):   # same module as the training kernels lucasboesen/name-that-llm-train-tokattn(2)
    def __init__(self, k, h=4096):
        super().__init__()
        self.drop = torch.nn.Dropout(0.2)
        self.proj = torch.nn.Linear(h, 64)
        self.score = torch.nn.Linear(64, 1)
        self.out = torch.nn.Linear(128, k)

    def forward(self, x, m):
        h = torch.tanh(self.proj(self.drop(x)))
        s = self.score(h).squeeze(-1).masked_fill(~m, -1e4)
        a = torch.softmax(s, -1).unsqueeze(-1)
        mf = m.unsqueeze(-1).float()
        mean = (h * mf).sum(1) / mf.sum(1).clamp(min=1)
        return self.out(self.drop(torch.cat([mean, (a * h).sum(1)], -1)))


@torch.no_grad()
def tokattn_proba(texts, head_file, backbone):
    hits = sorted(glob.glob('/kaggle/input/**/' + head_file, recursive=True))
    assert hits, f'attach the training-kernel output holding {head_file}'
    d = torch.load(hits[0], map_location='cpu')
    assert list(d['classes']) == LABELS
    decoder = d.get('decoder', True)
    if backbone == 'qwen3_5':
        path = None
        for c in sorted(glob.glob('/kaggle/input/**/config.json', recursive=True)):
            try:
                if json.load(open(c)).get('model_type') == 'qwen3_5':
                    path = os.path.dirname(c)
                    break
            except Exception:
                continue
    else:
        path = find_dir('deberta-v3-large')
    assert path, f'backbone {backbone} not attached'
    cfg = AutoConfig.from_pretrained(path)
    if decoder:
        tc = getattr(cfg, 'text_config', None) or cfg
        tc.num_hidden_layers = d['keep']
        if getattr(tc, 'layer_types', None):
            tc.layer_types = tc.layer_types[:d['keep']]
        lm = AutoModel.from_pretrained(path, config=cfg, dtype=torch.float16, device_map='auto').eval()
    else:
        lm = AutoModel.from_pretrained(path, dtype=torch.float32).to(DEV).eval()
    qtok = AutoTokenizer.from_pretrained(path)
    qtok.padding_side = 'right'
    if qtok.pad_token is None:
        qtok.pad_token = qtok.eos_token
    H = d.get('hidden', 4096)
    net = TokAttn(len(LABELS), H)
    net.load_state_dict(d['state_dict'])
    dev = lm.get_input_embeddings().weight.device
    net = net.to(dev).eval()
    mu, sd = d['mu'].float().to(dev), d['sd'].float().to(dev)
    ids = qtok(list(texts), add_special_tokens=not decoder, truncation=True, max_length=d['max_len'])['input_ids']
    P = np.zeros((len(texts), len(LABELS)))
    order = np.argsort([len(t) for t in ids])
    for b0 in range(0, len(order), TOKATTN['bs']):
        idx = order[b0:b0 + TOKATTN['bs']]
        batch = qtok.pad({'input_ids': [ids[i] for i in idx]}, return_tensors='pt').to(dev)
        out = lm(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'], output_hidden_states=True)
        hs = (out.hidden_states if getattr(out, 'hidden_states', None) is not None
              else out.language_model_outputs.hidden_states)[d['layer']]
        n = batch['attention_mask'].sum(1)
        T = int(n.max()) - 1
        x = torch.zeros(len(idx), max(T, 1), H, device=dev)
        m = torch.zeros(len(idx), max(T, 1), dtype=torch.bool, device=dev)
        for r in range(len(idx)):   # first token dropped, as in training (states stored in fp16 there)
            k = int(n[r]) - 1
            if k > 0:
                st = torch.nan_to_num(hs[r, 1:k + 1].to(dev).float()).half().float()
                x[r, :k] = ((st - mu) / sd).clamp(-8, 8)
                m[r, :k] = True
        P[idx] = torch.softmax(net(x, m), -1).cpu().numpy()
        del hs, out, batch
    del lm
    gc.collect()
    torch.cuda.empty_cache()
    return P


if TOKATTN['enabled'] and not SMOKE:
    for tkey, tfile, tbackbone in TOKATTN_HEADS:
        if len(TE) < 50:   # interactive run: sanity-check each head on train rows it was fit on (in-sample, must be high)
            pt = tokattn_proba(TR[:40], tfile, tbackbone)
            acc_t = float(np.mean(pt.argmax(1) == y[:40]))
            print(f'  {tkey} in-sample check on 40 train rows: acc {acc_t:.2f}', flush=True)
            assert acc_t > 0.5, f'{tkey} does not reproduce its training fit'
        add(tkey, tokattn_proba(TE, tfile, tbackbone))


# ------------------------------------------------ 4) fine-tuned encoder fold ensembles (softmax averaged over folds)
def find_folds(dir_name, model_type, local_fallbacks):
    """Fold dirs (fold0..fold4) under a Kaggle input dir named dir_name whose config has model_type."""
    roots = glob.glob(f'/kaggle/input/**/{dir_name}', recursive=True) + local_fallbacks
    for r in roots:
        fs = sorted(glob.glob(os.path.join(r, 'fold*')))
        fs = [f for f in fs if os.path.isdir(f) and os.path.exists(os.path.join(f, 'config.json'))
              and json.load(open(os.path.join(f, 'config.json'))).get('model_type') == model_type]
        if len(fs) == 5:
            return fs
    raise FileNotFoundError(f'no 5 {model_type} folds under {dir_name}')


@torch.no_grad()
def encoder_ensemble(folds, remote_code):
    etok = AutoTokenizer.from_pretrained(folds[0], trust_remote_code=remote_code)
    enc = etok(TE, truncation=True, max_length=512)['input_ids']
    order = np.argsort([len(e) for e in enc])
    P = np.zeros((len(TE), K))
    for fp in folds:
        em = AutoModelForSequenceClassification.from_pretrained(
            fp, trust_remote_code=remote_code, dtype=torch.float16 if DEV == 'cuda' else torch.float32,
            attn_implementation='sdpa').to(DEV).eval()
        cols = [LABELS.index(em.config.id2label[i]) for i in range(len(em.config.id2label))]
        for i in range(0, len(order), 32):
            idx = order[i:i + 32]
            b = etok.pad({'input_ids': [enc[j] for j in idx]}, return_tensors='pt').to(DEV)
            P[np.ix_(idx, cols)] += em(**b).logits.float().softmax(-1).cpu().numpy() / len(folds)
        del em
        gc.collect()
        if DEV == 'cuda':
            torch.cuda.empty_cache()
    return P


for key, dir_name, model_type, remote_code, local in ENCODERS:
    add(key, encoder_ensemble(find_folds(dir_name, model_type, local), remote_code))

# ------------------------------------------------ blend + transductive marginal balancing
# Sinkhorn column rescaling of the test posterior toward uniform class mass (alpha = partial strength).
# CV: never negative (in-domain +0.002..+0.007, also with unequal class counts); under genre shift (external
# data) predictions collapsed onto one class and balancing took balanced accuracy from 0.12 to 0.19.
P = np.exp(LOGIT - LOGIT.max(1, keepdims=True))
P /= P.sum(1, keepdims=True)
if BALANCE_ALPHA > 0 and len(P) >= 2:
    target = (1 - BALANCE_ALPHA) * P.sum(0) + BALANCE_ALPHA * np.full(K, len(P) / K)
    for _ in range(50):
        P *= (target / (P.sum(0) + 1e-12))[None, :]
        P /= P.sum(1, keepdims=True) + 1e-12
    print('balancing changed', int((P.argmax(1) != LOGIT.argmax(1)).sum()), 'of', len(P), flush=True)
pred = P.argmax(1)
if DC_TE is not None:
    _isc = np.isin(pred, CLAUDE_IDX)
    _s = np.log(np.clip(P[:, CLAUDE_IDX], 1e-12, 1)) + CLAUDE_GATE_LAM * DC_TE
    if LC_TE is not None:
        _s = _s + CLAUDE_LORA_LAM * LC_TE
    _new = CLAUDE_IDX[_s.argmax(1)]
    print('claude gate changed', int((_new[_isc] != pred[_isc]).sum()), 'of', int(_isc.sum()), 'Claude predictions', flush=True)
    pred[_isc] = _new[_isc]
sub = pd.DataFrame({'id': test['id'], 'model': [LABELS[i] for i in pred]})
assert sub['id'].is_unique and sub['model'].isin(LABELS).all() and len(sub) == len(test)
sub.to_parquet('submission.parquet', index=False)
print(sub['model'].value_counts().to_string())
print(f'total {time.time() - T0:.0f}s')


### Claude decision and temporary gate state


In [ ]:
%%writefile /kaggle/temp/ntl-v6-pfn184-code/claude_gate.py
"""Exact ntl-v6 hard Claude decision, with lossless temporary state transport."""
from pathlib import Path
import numpy as np


def validate_gate(P, labels, DC_TE, LC_TE, CLAUDE_IDX, gate_lam, lora_lam):
    labels = np.asarray(labels)
    assert labels.ndim == 1 and len(labels) == len(np.unique(labels)) == 18
    assert P.dtype == np.float64 and P.shape[1:] == (len(labels),)
    assert np.isfinite(P).all() and (P >= 0).all()
    expected = np.array([i for i, label in enumerate(labels) if label.startswith('claude')])
    assert len(expected) == 5
    assert np.issubdtype(CLAUDE_IDX.dtype, np.integer)
    np.testing.assert_array_equal(CLAUDE_IDX, expected)
    assert (gate_lam, lora_lam) == (1.5, 1.0)
    for value in (DC_TE, LC_TE):
        if value is not None:
            assert np.issubdtype(value.dtype, np.floating)
            assert value.shape == (len(P), len(CLAUDE_IDX)) and np.isfinite(value).all()


def gated_predict(P, labels, DC_TE, LC_TE, CLAUDE_IDX, gate_lam=1.5, lora_lam=1.0):
    validate_gate(P, labels, DC_TE, LC_TE, CLAUDE_IDX, gate_lam, lora_lam)
    # Keep the reference operations, dtypes, clipping, tie order and optional gate exactly.
    pred = P.argmax(1)
    if DC_TE is not None:
        _isc = np.isin(pred, CLAUDE_IDX)
        _s = np.log(np.clip(P[:, CLAUDE_IDX], 1e-12, 1)) + gate_lam * DC_TE
        if LC_TE is not None:
            _s = _s + lora_lam * LC_TE
        _new = CLAUDE_IDX[_s.argmax(1)]
        pred[_isc] = _new[_isc]
    return pred


def save_gate_state(directory, P, ids, labels, DC_TE, LC_TE, CLAUDE_IDX, gate_lam, lora_lam):
    validate_gate(P, labels, DC_TE, LC_TE, CLAUDE_IDX, gate_lam, lora_lam)
    # Explicit flags avoid object arrays and preserve the actual present-array dtypes.
    np.savez_compressed(Path(directory)/'claude_gate_state.npz',
        ids=np.asarray(ids, dtype=str), labels=np.asarray(labels, dtype=str),
        dc_present=np.asarray(DC_TE is not None), lc_present=np.asarray(LC_TE is not None),
        DC_TE=DC_TE if DC_TE is not None else np.empty((0, 5), dtype=np.float64),
        LC_TE=LC_TE if LC_TE is not None else np.empty((0, 5), dtype=np.float64),
        CLAUDE_IDX=CLAUDE_IDX, gate_lam=np.asarray(gate_lam), lora_lam=np.asarray(lora_lam))


def load_gate_state(directory, P, ids, labels):
    with np.load(Path(directory)/'claude_gate_state.npz', allow_pickle=False) as saved:
        np.testing.assert_array_equal(saved['ids'], ids)
        np.testing.assert_array_equal(saved['labels'], labels)
        state = dict(DC_TE=saved['DC_TE'].copy() if bool(saved['dc_present']) else None,
            LC_TE=saved['LC_TE'].copy() if bool(saved['lc_present']) else None,
            CLAUDE_IDX=saved['CLAUDE_IDX'].copy(),
            gate_lam=saved['gate_lam'].item(), lora_lam=saved['lora_lam'].item())
    validate_gate(P, labels, **state)
    return state


### Observation-only feature and gate capture


In [ ]:
%%writefile /kaggle/temp/ntl-v6-pfn184-code/capture_runtime.py
"""Capture existing frozen features and balanced baseline into ephemeral storage."""
import hashlib
import json
import os
from functools import wraps
from pathlib import Path
import numpy as np
from claude_gate import save_gate_state

VIEWS = {'q3_8b_cat7+22':8192, 'style4b':5120}

def cache_directory():
    directory = Path(os.environ['NTL_TABULAR_TEMP']).resolve()
    assert directory.is_relative_to(Path('/kaggle/temp').resolve())
    directory.mkdir(parents=True,exist_ok=True)
    return directory

def wrap_fit_head(original, train, test, labels):
    cache = cache_directory()
    np.savez_compressed(cache/'alignment.npz',
        train_ids=train.id.to_numpy(dtype=str), test_ids=test.id.to_numpy(dtype=str),
        labels=np.asarray(labels,dtype=str),
        y=train.model.map({label:i for i,label in enumerate(labels)}).to_numpy(dtype=np.int64))
    captured = set()
    @wraps(original)
    def fitting(key, Etr, Ete, C):
        if key in VIEWS:
            assert key not in captured
            assert Etr.dtype==Ete.dtype==np.float32
            assert Etr.shape==(len(train),VIEWS[key]) and Ete.shape==(len(test),VIEWS[key])
            assert np.isfinite(Etr).all() and np.isfinite(Ete).all()
            for suffix,value in [('train',Etr),('test',Ete)]:
                np.save(cache/(key+'_'+suffix+'.npy'),value,allow_pickle=False)
            captured.add(key)
            (cache/'captured_views.json').write_text(json.dumps(sorted(captured)))
        return original(key,Etr,Ete,C)
    return fitting

def capture_balanced(P,test,labels,DC_TE,LC_TE,CLAUDE_IDX,gate_lam,lora_lam):
    cache=cache_directory()
    assert set(json.loads((cache/'captured_views.json').read_text()))==set(VIEWS)
    assert P.dtype==np.float64 and P.shape==(len(test),len(labels))
    assert np.isfinite(P).all() and (P>=0).all()
    np.savez_compressed(cache/'balanced_baseline.npz',probabilities=P,
        ids=test.id.to_numpy(dtype=str),labels=np.asarray(labels,dtype=str))
    save_gate_state(cache,P,test.id.to_numpy(dtype=str),labels,
        DC_TE,LC_TE,CLAUDE_IDX,gate_lam,lora_lam)
    records={}
    for path in sorted(cache.glob('*.npy')):
        with path.open('rb') as stream: digest=hashlib.file_digest(stream,'sha256').hexdigest()
        records[path.name]=dict(bytes=path.stat().st_size,sha256=digest)
    # Only sizes and hashes are exportable; arrays and targets stay under /kaggle/temp.
    Path('/kaggle/working/tabular_capture_audit.json').write_text(json.dumps(dict(
        status='TWO_EXISTING_VIEWS_CAPTURED_WITHOUT_EXTRA_BACKBONE_PASS',
        train_rows=len(np.load(cache/'alignment.npz',allow_pickle=False)['train_ids']),
        query_rows=len(test),views=records,baseline_classes=list(labels),
        source='exact ntl-v6 v2 raw fit_head inputs before preprocessing',
        gate_state_captured_before_original_argmax=True,
        dc_dtype=None if DC_TE is None else str(DC_TE.dtype),
        lc_dtype=None if LC_TE is None else str(LC_TE.dtype),
        claude_indices_dtype=str(CLAUDE_IDX.dtype),
        ephemeral_training_arrays=True),indent=2))


### Original reference execution and gated control


In [ ]:
%%writefile /kaggle/temp/ntl-v6-pfn184-code/run_baseline.py
"""Execute the exact path-repaired ntl-v6 reference with two observation hooks."""
import hashlib
import json
import os
from pathlib import Path
from capture_runtime import wrap_fit_head, capture_balanced

assert os.name != 'nt' and Path('/kaggle/input').is_dir()
here = Path(__file__).resolve().parent
config = json.loads((here/'runtime_config.json').read_text())
raw = (here/'ntl_reference.py').read_bytes()
assert hashlib.sha256(raw).hexdigest() == config['reference_source_sha256']
source = raw.decode('utf-8')
assert hashlib.sha256((here/'centered_stylegram_adapter.py').read_bytes()).hexdigest()==config['centered_stylegram_adapter_sha256']
from centered_stylegram_adapter import centered_stylegram_log_evidence
balance_point='BALANCE_ALPHA = 0.5'
balance_override='BALANCE_ALPHA = 0.25'
style_point="add('stylegram', p)"
assert source.count(balance_point)==source.count(style_point)==1
assert config['baseline_balance_alpha']==.25 and config['centered_stylegram_shrinkage']==1.0
style_hook="""_style_evidence, _style_diagnostics = _centered_stylegram_log_evidence(
    hstack(A, format='csr'), hstack(B, format='csr'), y, train['prompt_id'].values, TE, p,
    shrinkage=1.0, adapt_enabled=ADAPT)
assert _style_evidence.shape == LOGIT.shape and np.isfinite(_style_evidence).all()
LOGIT[:] += W['stylegram'] * _style_evidence
_style_diagnostics.update(stylegram_weight=float(W['stylegram']), direct_log_evidence=True,
    plain_not_double_counted=True, svc_C=.5, svc_random_state=42, svc_max_iter=4000,
    balance_alpha=.25, original_component_weights_unchanged=True)
pathlib.Path('/kaggle/working/centered_stylegram_audit.json').write_text(json.dumps(_style_diagnostics, indent=2))
print(f'stylegram done ({time.time() - T0:.0f}s)', flush=True)
del _style_evidence"""
loop = 'for key, mt, hid, layers, C, smoke_id, pool in LLM_HEADS:'
prediction = 'pred = P.argmax(1)'
assert source.count(loop) == source.count(prediction) == 1
capture_fit = 'fit_head = _tabular_wrap_fit_head(fit_head, train, test, LABELS)\n' + loop
capture_probability = ('_tabular_capture_balanced(P, test, LABELS, DC_TE, LC_TE, '
    'CLAUDE_IDX, CLAUDE_GATE_LAM, CLAUDE_LORA_LAM)\n' + prediction)
modified = source.replace(balance_point,balance_override).replace(style_point,style_hook).replace(loop,capture_fit).replace(prediction,capture_probability)
assert modified.replace(balance_override,balance_point).replace(style_hook,style_point).replace(capture_fit,loop).replace(capture_probability,prediction)==source
Path('/kaggle/working/ntl_v6_source_audit.json').write_text(json.dumps(dict(
    status='NTL_V6_ORIGINAL_WEIGHTS_CENTERED_STYLEGRAM_ALPHA025_AND_CAPTURE_HOOKS',
    source_notebook=config['reference_notebook'], source_version=2,
    scored_source_version=1, scored_script_version_id=config['reference_scored_script_version_id'],
    original_source_sha256=hashlib.sha256(raw).hexdigest(),
    executed_source_sha256=hashlib.sha256(modified.encode('utf-8')).hexdigest(),
    original_scientific_functions_unchanged=True, weights_unchanged=True,balance_alpha=.25,stylegram_shrinkage=1.0,
    original_claude_gate_unchanged=True, extra_backbone_passes=0), indent=2))
exec(compile(modified, 'ntl_v6_with_observation_hooks', 'exec'), dict(
    __name__='__main__', __file__=str(here/'ntl_reference.py'),
    _tabular_wrap_fit_head=wrap_fit_head, _tabular_capture_balanced=capture_balanced,_centered_stylegram_log_evidence=centered_stylegram_log_evidence))
assert Path('/kaggle/working/submission.parquet').is_file()


### Train-only masked sequence features


In [ ]:
%%writefile /kaggle/temp/ntl-v6-pfn184-code/masked_features.py
"""Fixed shape/function TF-IDF-SVD blocks; all learned transforms fit on train only."""

import re

import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.decomposition import TruncatedSVD

FUNCTION_WORDS_TEXT='a an the and or but if while although because as so than that which who whom whose this these those it its they them their theirs he him his she her hers we us our ours you your yours i me my mine in on at to from with without for of by over under between through into about is are was were be been being have has had do does did can could will would shall should may might must not no nor all any some each every both either neither other another such only also just very too more most less least here there when where why how what whether then now still yet even much many few little'

def shape_text(text):
    # Preserve character count/case, punctuation and whitespace; suppress letters/digits.
    return re.sub(r'\d','#',re.sub(r'[^\W\d_]',lambda m:'A' if m[0].isupper() else 'a',text))

def function_text(text):
    words=set(FUNCTION_WORDS_TEXT.split())
    return ' '.join(t.lower() if t.lower() in words else ('CONTENT' if t.isalnum() else t)
        for t in re.findall(r'\w+|[^\w\s]',text))

VECTORIZERS = {'shape_char': {'min_df': 3, 'max_df': 0.995, 'sublinear_tf': True, 'norm': 'l2', 'analyzer': 'char', 'ngram_range': [3, 5], 'lowercase': False, 'max_features': 20000}, 'function_word': {'min_df': 3, 'max_df': 0.995, 'sublinear_tf': True, 'norm': 'l2', 'analyzer': 'word', 'ngram_range': [1, 3], 'lowercase': False, 'token_pattern': '(?u)\\b\\w+\\b|[^\\w\\s]', 'max_features': 20000}}

def masked_svd_blocks(train_texts, query_texts):
    train_blocks, query_blocks, audit = [], [], []
    for name, cfg in VECTORIZERS.items():
        masker = shape_text if name == 'shape_char' else function_text
        fit_values = [masker(text) for text in train_texts]
        query_values = [masker(text) for text in query_texts]
        options = dict(cfg)
        options['ngram_range'] = tuple(options['ngram_range'])
        options['dtype'] = np.float32
        vectorizer = TfidfVectorizer(**options)
        sparse_fit = vectorizer.fit_transform(fit_values)
        assert sparse_fit.shape[1] > 16
        svd = TruncatedSVD(n_components=16, n_iter=5, random_state=20261017)
        svd.fit(sparse_fit)
        train_block = svd.transform(vectorizer.transform(fit_values)).astype(np.float32)
        query_block = svd.transform(vectorizer.transform(query_values)).astype(np.float32)
        assert train_block.shape == (len(train_texts), 16)
        assert query_block.shape == (len(query_texts), 16)
        assert np.isfinite(train_block).all() and np.isfinite(query_block).all()
        train_blocks.append(train_block)
        query_blocks.append(query_block)
        audit.append(dict(view=name, vocabulary_size=len(vectorizer.vocabulary_),
            fit_rows=len(train_texts), query_rows=len(query_texts), components=16,
            fit_scope='official_train_only',
            explained_variance_ratio_sum=float(svd.explained_variance_ratio_.sum())))
    return np.concatenate(train_blocks, axis=1), np.concatenate(query_blocks, axis=1), audit



### TabPFN complement

In [ ]:
%%writefile /kaggle/temp/ntl-v6-pfn184-code/tabular_worker.py
"""Fixed train-only feature preparation or one isolated tabular inference member."""
import hashlib
import importlib.metadata
import json
import os
import re
import sys
import time
from pathlib import Path

assert os.name!='nt' and Path('/kaggle/input').is_dir()
for name in ('OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS'):os.environ[name]='4'
os.environ['HF_HUB_OFFLINE']='1'
import numpy as np

here=Path(__file__).resolve().parent
config=json.loads((here/'runtime_config.json').read_text())
cache=Path(os.environ['NTL_TABULAR_TEMP']).resolve()
assert cache.is_relative_to(Path('/kaggle/temp').resolve())
mode=sys.argv[1]
started=time.monotonic()

def sha(path):
    with path.open('rb') as stream:return hashlib.file_digest(stream,'sha256').hexdigest()

def style24(t):
    # Original reference first 20 deterministic scalars, plus four fixed ratios.
    words = re.findall(r"[A-Za-z\u2019']+", t)
    nw, nc = max(len(words), 1), max(len(t), 1)
    sents = [x for x in re.split(r'(?<=[.!?])\s+', t.strip()) if x]
    sl = np.array([len(re.findall(r"[A-Za-z\u2019']+", x)) for x in sents]) if sents else np.array([0])
    wl = np.array([len(x) for x in words]) if words else np.array([0])
    lw = [x.lower() for x in words]
    return [np.log1p(nw), len(sents), sl.mean(), sl.std(), sl.max(), sl.min(), wl.mean(), wl.std(),
        (wl >= 9).mean(), len(set(lw))/nw, sum(c.isupper() for c in t)/nc, sum(c.isdigit() for c in t)/nc,
        sum(ord(c)>127 for c in t)/nc, t.count(', ')/nw, t.count(' and ')/nw, t.count(' but ')/nw,
        sum(x[:1].isupper() and x.split()[0].lower() in ('this','these','it','they') for x in sents)/max(len(sents),1),
        sum(x.endswith('ly') for x in words)/nw, sum(x.endswith('ing') for x in words)/nw,
        sum(x.endswith('tion') for x in words)/nw,
        float(any(c in t for c in ('\u2019','\u201c','\u201d'))), t.count('\u2014')/nw,
        t.count('\n')/nc, sum(bool(x.strip()) for x in t.splitlines())/nw]

with np.load(cache/'alignment.npz',allow_pickle=False) as saved:
    alignment={key:saved[key].copy() for key in saved.files}
if mode=='prepare':
    import pandas as pd
    from sklearn.decomposition import PCA
    from threadpoolctl import threadpool_limits
    threadpool_limits(4)
    data=next(path for path in [Path('/kaggle/input/competitions/name-that-llm'),Path('/kaggle/input/name-that-llm')] if (path/'train.csv').is_file())
    train,test=pd.read_csv(data/'train.csv'),pd.read_csv(data/'test.csv')
    for frame,key in [(train,'train_ids'),(test,'test_ids')]:
        np.testing.assert_array_equal(frame.id.to_numpy(dtype=str),alignment[key])
    np.testing.assert_array_equal(train.model.to_numpy(dtype=str),alignment['labels'][alignment['y']])
    train_columns,test_columns,audit=[],[],[]
    for key,width in [('q3_8b_cat7+22',8192),('style4b',5120)]:
        train_view=np.load(cache/(key+'_train.npy'),mmap_mode='r',allow_pickle=False)
        test_view=np.load(cache/(key+'_test.npy'),mmap_mode='r',allow_pickle=False)
        assert train_view.shape==(len(train),width) and test_view.shape==(len(test),width)
        pca=PCA(n_components=64,svd_solver='randomized',iterated_power=3,random_state=42)
        pca.fit(train_view)
        train_columns.append(pca.transform(train_view).astype(np.float32))
        test_columns.append(pca.transform(test_view).astype(np.float32))
        audit.append(dict(view=key,fit_rows=len(train),components=64,fit_scope='official_train_only',
            explained_variance=float(pca.explained_variance_ratio_.sum())))
    for frame,columns in [(train,train_columns),(test,test_columns)]:
        texts=frame.text.fillna('').astype(str)
        columns.append(np.asarray([style24(text) for text in texts],dtype=np.float32))
    assert sha(here/'masked_features.py')==config['masked_feature_module_sha256']
    base_train,base_test=np.concatenate(train_columns,axis=1),np.concatenate(test_columns,axis=1)
    from masked_features import masked_svd_blocks
    text_train,text_test,text_audit=masked_svd_blocks(
        train.text.fillna('').astype(str).tolist(),test.text.fillna('').astype(str).tolist())
    train_columns.append(text_train)
    test_columns.append(text_test)
    x_train,x_test=np.concatenate(train_columns,axis=1),np.concatenate(test_columns,axis=1)
    assert x_train.shape==(len(train),184) and x_test.shape==(len(test),184)
    np.testing.assert_array_equal(x_train[:,:152],base_train)
    np.testing.assert_array_equal(x_test[:,:152],base_test)
    assert np.isfinite(x_train).all() and np.isfinite(x_test).all()
    np.savez(cache/'tables.npz',x_train=x_train,x_test=x_test,**alignment)
    Path('/kaggle/working/tabular_prepare_audit.json').write_text(json.dumps(dict(
        status='FULL_TRAIN_ONLY_PCA_STYLE24_AND_MASKED_SVD_COMPLETE',pca=audit,text_svd=text_audit,features=184,
        masked_feature_module_sha256=config['masked_feature_module_sha256'],
        query_rows=len(test),elapsed_seconds=time.monotonic()-started,
        packages={name:importlib.metadata.version(name) for name in ('numpy','pandas','scikit-learn','scipy')},
        training_context_exported=False),indent=2))
    sys.exit(0)

assert mode=='tabpfn'
import torch
assert torch.cuda.is_available()
assert torch.cuda.mem_get_info()[0]>8*1024**3
with np.load(cache/'tables.npz',allow_pickle=False) as saved:
    x_train,x_test,y=(saved[key].copy() for key in ('x_train','x_test','y'))
asset=Path(os.environ['NTL_TABULAR_ASSET'])
settings=config[mode]
checkpoint=asset/settings['checkpoint_filename']
assert checkpoint.is_file() and sha(checkpoint)==settings['checkpoint_sha256']
assert importlib.metadata.version(mode)==settings['package_version']
assert settings['n_estimators']==128,'Preserve the selected128-estimator configuration'
from tabpfn import TabPFNClassifier
from tabpfn.constants import ModelVersion
params=dict(n_estimators=128,model_path=str(checkpoint),device='cuda:0',fit_mode='low_memory',
    memory_saving_mode='auto',balance_probabilities=True,tuning_config=None,random_state=42,n_preprocessing_jobs=1)
model=TabPFNClassifier.create_default_for_version(ModelVersion.V3_5,**params)
torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize();t=time.monotonic()
model.fit(x_train,y)
torch.cuda.synchronize();fit_seconds=time.monotonic()-t
t=time.monotonic();raw=model.predict_proba(x_test)
torch.cuda.synchronize();predict_seconds=time.monotonic()-t
classes=np.asarray(model.classes_,dtype=int)
np.testing.assert_array_equal(np.sort(classes),np.arange(18))
probabilities=np.zeros((len(x_test),18),dtype=np.float64)
probabilities[:,classes]=raw
assert np.isfinite(probabilities).all() and (probabilities>=0).all()
np.testing.assert_allclose(probabilities.sum(1),1,rtol=0,atol=1e-5)
np.savez(cache/(mode+'_proba.npz'),proba=probabilities,ids=alignment['test_ids'],labels=alignment['labels'])
Path('/kaggle/working/'+mode+'_inference_audit.json').write_text(json.dumps(dict(
    status='COMPLETE',model=mode,checkpoint_sha256=settings['checkpoint_sha256'],parameters=params,
    train_rows=len(x_train),query_rows=len(x_test),features=184,fit_seconds=fit_seconds,
    predict_seconds=predict_seconds,peak_cuda_allocated_bytes=int(torch.cuda.max_memory_allocated()),
    peak_cuda_reserved_bytes=int(torch.cuda.max_memory_reserved()),probability_transform='none',
    row_mass_maxabs=float(np.max(np.abs(probabilities.sum(1)-1))),
    packages={name:importlib.metadata.version(name) for name in (mode,'torch','numpy','scikit-learn')},
    oof=False,model_context_exported=False),indent=2))


### Fixed punctuation correction


In [ ]:
%%writefile /kaggle/temp/ntl-v6-pfn184-code/orthography_post.py
"""Fixed local orthographic correction after original marginal balancing.

The two features, family masks and penalty 2 credit saddamaditya's Apex v7:
https://www.kaggle.com/code/saddamaditya/name-that-llm-apex-stylometric-centering?kernelVersion=7
The correction follows the posterior fusion and precedes the original Claude gate.
"""
import json
from pathlib import Path
import numpy as np

def fixed_penalty(texts):
    curly=np.asarray([any(c in t for c in ('\u2019','\u201c','\u201d')) for t in texts])
    em2=np.asarray([t.count('\u2014')>=2 for t in texts])
    cg=np.asarray([c.startswith('claude') or c.startswith('glm') for c in classes])
    ge=np.asarray([c.startswith('gemini') for c in classes])
    return -2.0*(curly[:,None]*cg[None]+em2[:,None]*ge[None]),curly|em2

def post_balance(logp,penalty,triggered):
    # Preserve untouched log probabilities and their probabilities exactly.
    result=logp.copy()
    values=logp[triggered]+penalty[triggered]
    p=np.exp(values-values.max(1,keepdims=True))
    p/=p.sum(1,keepdims=True)
    result[triggered]=np.log(p.clip(1e-300))
    np.testing.assert_array_equal(result[~triggered],logp[~triggered])
    np.testing.assert_array_equal(np.exp(result[~triggered]),np.exp(logp[~triggered]))
    return result


def apply_post_balance(probability, texts, labels):
    """Adapt native probabilities to the exact tested function; keep untouched rows exact."""
    global classes
    classes = np.asarray(labels, dtype=str)
    assert len(classes) == len(np.unique(classes)) == 18
    assert probability.dtype == np.float64 and probability.shape == (len(texts), len(classes))
    assert np.isfinite(probability).all() and (probability >= 0).all()
    cg = np.asarray([c.startswith('claude') or c.startswith('glm') for c in classes])
    ge = np.asarray([c.startswith('gemini') for c in classes])
    assert not (cg & ge).any()
    penalty, triggered = fixed_penalty(list(texts))
    tested_result = post_balance(np.log(probability.clip(1e-300)), penalty, triggered)
    result = probability.copy()
    # Assign ONLY triggered rows: no exp(log(P)) roundtrip for untouched rows.
    result[triggered] = np.exp(tested_result[triggered])
    np.testing.assert_array_equal(result[~triggered], probability[~triggered])
    assert np.isfinite(result).all() and (result >= 0).all()
    # Raw tabular probabilities retain their tested floating-point row mass.
    # The operator is unchanged: only triggered rows are renormalized.
    np.testing.assert_allclose(result[triggered].sum(1), 1, rtol=0, atol=2e-12)
    np.testing.assert_array_equal(result[~triggered].sum(1), probability[~triggered].sum(1))
    audit = dict(status='FIXED_POST_BALANCE_ROW_CORRECTION_APPLIED',
        source_notebook='saddamaditya/name-that-llm-apex-stylometric-centering', source_version=7,
        tested_cpu_script_version_id=359125645, penalty=2.0, rows=len(result),
        triggered_rows=int(triggered.sum()), untouched_rows_bitexact=True,
        class_masks_disjoint=True, second_column_balance=False,
        prediction_changes=int((result.argmax(1) != probability.argmax(1)).sum()))
    Path('/kaggle/working/orthography_post_balance_audit.json').write_text(json.dumps(audit, indent=2))
    print(json.dumps(audit), flush=True)
    return result


### Geometric fusion, punctuation, then hard Claude gate

In [ ]:
%%writefile /kaggle/temp/ntl-v6-pfn184-code/final_pipeline.py
"""Geometric PFN fusion, small TabICL complement, punctuation, then original gate."""
import numpy as np
from claude_gate import gated_predict
from orthography_post import apply_post_balance


def geometric_pool(base, tabular, tabpfn_exponent=.2):
    """Pool raw class-aligned probabilities with exponents; normalize once by row."""
    assert base.dtype == tabular.dtype == np.float64
    assert base.ndim == 2 and base.shape == tabular.shape and base.shape[1] == 18
    assert np.isfinite(base).all() and np.isfinite(tabular).all()
    assert (base >= 0).all() and (tabular >= 0).all()
    assert (base.sum(1) > 0).all() and (tabular.sum(1) > 0).all()
    np.testing.assert_allclose(tabular.sum(1), 1, rtol=0, atol=1e-5)
    assert tabpfn_exponent == .2
    a = tabpfn_exponent
    values = (1-a)*np.log(np.clip(base, 1e-300, None)) + a*np.log(np.clip(tabular, 1e-300, None))
    mixed = np.exp(values-values.max(1, keepdims=True))
    mixed /= mixed.sum(1, keepdims=True)
    assert np.isfinite(mixed).all() and (mixed >= 0).all()
    np.testing.assert_allclose(mixed.sum(1), 1, rtol=0, atol=2e-15)
    return mixed


def combine_complements(base, tabular, tabicl):
    """Preserve raw TabICL probability mass in the fixed .95/.05 arithmetic mix."""
    assert tabicl.dtype == np.float64 and tabicl.shape == base.shape
    assert np.isfinite(tabicl).all() and (tabicl >= 0).all()
    np.testing.assert_allclose(tabicl.sum(1), 1, rtol=0, atol=1e-5)
    geometric = geometric_pool(base, tabular, tabpfn_exponent=.2)
    mixed = .95*geometric + .05*tabicl
    expected_mass = .95*geometric.sum(1) + .05*tabicl.sum(1)
    np.testing.assert_allclose(mixed.sum(1), expected_mass, rtol=0, atol=2e-15)
    return mixed, expected_mass


def combine_and_predict(base, tabular, texts, labels, gate, tabicl):
    assert base.shape == tabular.shape == tabicl.shape == (len(texts), len(labels))
    mixed, expected_mass = combine_complements(base, tabular, tabicl)
    corrected = apply_post_balance(mixed, texts, labels)
    np.testing.assert_allclose(corrected.sum(1), 1, rtol=0, atol=1e-5)
    pred = gated_predict(corrected, labels, **gate)
    return pred, corrected, expected_mass


### Aligned final predictions and original-control replay


In [ ]:
%%writefile /kaggle/temp/ntl-v6-pfn184-code/finish_prediction.py
"""Finalize the own complement while retaining the reference's last Claude decision."""
import hashlib
import json
import os
from pathlib import Path
import numpy as np
import pandas as pd
from claude_gate import gated_predict, load_gate_state
from final_pipeline import combine_and_predict

assert os.name != 'nt' and Path('/kaggle/input').is_dir()
here = Path(__file__).resolve().parent
config = json.loads((here/'runtime_config.json').read_text())
cache = Path(os.environ['NTL_TABULAR_TEMP']).resolve()
assert cache.is_relative_to(Path('/kaggle/temp').resolve())
assert config['fusion'] == dict(kind='normalized_geometric', base_exponent=.8,
    tabpfn_exponent=.2, probability_floor=1e-300, input_probability_transform='none')
assert config['baseline_balance_alpha'] == .25 and config['tabpfn']['n_estimators'] == 128
assert config['tabicl']['n_estimators'] == 64
assert config['tabicl_mix'] == dict(geometric_weight=.95, tabicl_weight=.05,
    probability_transform='none', row_renormalized=False)
with np.load(cache/'balanced_baseline.npz', allow_pickle=False) as saved:
    base, ids, labels = (saved[key].copy() for key in ('probabilities', 'ids', 'labels'))
with np.load(cache/'tabpfn_proba.npz', allow_pickle=False) as saved:
    np.testing.assert_array_equal(saved['ids'], ids)
    np.testing.assert_array_equal(saved['labels'], labels)
    tabular = saved['proba'].copy()
with np.load(cache/'tabicl_proba.npz', allow_pickle=False) as saved:
    np.testing.assert_array_equal(saved['ids'], ids)
    np.testing.assert_array_equal(saved['labels'], labels)
    tabicl = saved['proba'].copy()
gate = load_gate_state(cache, base, ids, labels)
data = next(path for path in [Path('/kaggle/input/competitions/name-that-llm'),
    Path('/kaggle/input/name-that-llm')] if (path/'test.csv').is_file())
test = pd.read_csv(data/'test.csv')
np.testing.assert_array_equal(test.id.to_numpy(dtype=str), ids)
control = pd.read_parquet('/kaggle/working/ntl_v6_gated_control.parquet')
np.testing.assert_array_equal(control.id.to_numpy(dtype=str), ids)
# Verify transport against the ORIGINAL reference's actual gated labels, not P.argmax.
np.testing.assert_array_equal(control.model.to_numpy(dtype=str), labels[gated_predict(base, labels, **gate)])
pred, corrected, expected_mass = combine_and_predict(base, tabular,
    test.text.fillna('').astype(str).tolist(), labels, gate, tabicl=tabicl)
submission = pd.DataFrame(dict(id=test.id, model=labels[pred]))
assert len(submission) == len(test) and submission.id.is_unique and submission.model.isin(labels).all()
submission.to_parquet('/kaggle/working/submission.parquet', index=False)
with Path('/kaggle/working/submission.parquet').open('rb') as stream:
    digest = hashlib.file_digest(stream, 'sha256').hexdigest()
Path('/kaggle/working/tabular_combination_audit.json').write_text(json.dumps(dict(
    status='NTL_V6_GEOMETRIC_PFN_TABICL_POST_AND_FINAL_CLAUDE_GATE_COMPLETE',
    baseline=dict(notebook=config['reference_notebook'], source_version=2,
        scored_version=1, scored_script_version_id=config['reference_scored_script_version_id']),
    fusion=dict(kind='normalized_geometric',exponents=dict(base=.8,tabpfn=.2),
        probability_floor=1e-300,row_normalized=True),
    tabicl_mix=dict(geometric_weight=.95,tabicl_weight=.05,probability_transform='none',row_renormalized=False),
    tabicl=dict(checkpoint=config['tabicl']['checkpoint_filename'],n_estimators=config['tabicl']['n_estimators'],
        norm_methods=['quantile'],softmax_temperature=.9),
    balance_alpha=.25,n_estimators=128,centered_stylegram_shrinkage=1.0, query_rows=len(test), classes=labels.tolist(),
    input_probability_transform='none', second_column_balance=False,
    operator_order=['reference_sinkhorn_alpha_025', 'normalized_geometric_pfn_exponent_02', 'raw_tabicl_probability_mix_005', 'punctuation_penalty_2', 'reference_hard_claude_gate'],
    dc_present=gate['DC_TE'] is not None, lc_present=gate['LC_TE'] is not None,
    baseline_gate_replay_bitexact=True,
    prediction_changes_vs_gated_control=int((submission.model.to_numpy() != control.model.to_numpy()).sum()),
    final_gate_changes=int((pred != corrected.argmax(1)).sum()),
    geometric_fusion_row_normalized=True,
    expected_mixture_row_mass_maxabs=float(np.max(np.abs(expected_mass-1))),
    final_corrected_row_mass_maxabs=float(np.max(np.abs(corrected.sum(1)-1))),
    submission_sha256=digest, baseline_control='ntl_v6_gated_control.parquet',
    training_context_exported=False, oof=False), indent=2))


### Isolated quantile TabICL complement

In [ ]:
%%writefile /kaggle/temp/ntl-v6-pfn184-code/tabicl_worker.py
"""Isolated offline TabICL v1.1 quantile inference on the unchanged 152-column prefix."""
import os
assert os.name != 'nt' and os.path.isdir('/kaggle/input'), 'Kaggle execution only'
for key in ('OMP_NUM_THREADS','MKL_NUM_THREADS','OPENBLAS_NUM_THREADS'):
    os.environ[key] = '4'
os.environ.update(CUDA_VISIBLE_DEVICES='0', HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1')
import hashlib
import importlib.metadata as metadata
import json
import time
from pathlib import Path
import numpy as np
import torch
from tabicl import TabICLClassifier
from threadpoolctl import threadpool_limits

threadpool_limits(4)
torch.set_num_threads(4)
here = Path(__file__).resolve().parent
config = json.loads((here/'runtime_config.json').read_text())
settings = config['tabicl']
cache = Path(os.environ['NTL_TABULAR_TEMP']).resolve()
assert cache.is_relative_to(Path('/kaggle/temp').resolve())
with np.load(cache/'tables.npz', allow_pickle=False) as saved:
    train184, query184, y, ids, labels = (saved[k].copy() for k in ('x_train','x_test','y','test_ids','labels'))
with np.load(cache/'alignment.npz', allow_pickle=False) as saved:
    np.testing.assert_array_equal(saved['y'], y)
    np.testing.assert_array_equal(saved['test_ids'], ids)
    np.testing.assert_array_equal(saved['labels'], labels)
assert train184.shape[1] == query184.shape[1] == 184
fit_x, query_x = train184[:, :152].copy(), query184[:, :152].copy()
np.testing.assert_array_equal(fit_x, train184[:, :152])
np.testing.assert_array_equal(query_x, query184[:, :152])
assert fit_x.dtype == query_x.dtype == np.float32
assert np.isfinite(fit_x).all() and np.isfinite(query_x).all()
np.testing.assert_array_equal(np.unique(y), np.arange(18))
assert metadata.version('tabicl') == settings['package_version'] == '2.2.0'
assert metadata.version('einops') == '0.8.2'
assert torch.cuda.is_available()
asset = Path(os.environ['NTL_TABICL_ASSET'])
checkpoint = asset/settings['checkpoint_filename']
assert checkpoint.is_file() and checkpoint.stat().st_size == settings['checkpoint_bytes']
with checkpoint.open('rb') as stream:
    assert hashlib.file_digest(stream, 'sha256').hexdigest() == settings['checkpoint_sha256']
assert settings['n_estimators'] == 64, 'Frozen selected quantile64 configuration'
params = dict(checkpoint_version=checkpoint.name,model_path=str(checkpoint),allow_auto_download=False,
    n_estimators=settings['n_estimators'],norm_methods=['quantile'],softmax_temperature=.9,
    average_logits=False,support_many_classes=True,batch_size=4,kv_cache=False,device='cuda:0',
    use_amp=True,use_fa3=False,offload_mode='auto',random_state=777,n_jobs=4)
free, total = torch.cuda.mem_get_info()
assert free > 8*1024**3
torch.cuda.reset_peak_memory_stats()
model = TabICLClassifier(**params)
torch.cuda.synchronize(); before = time.monotonic()
model.fit(fit_x, y)
torch.cuda.synchronize(); fit_seconds = time.monotonic()-before
before = time.monotonic(); raw = model.predict_proba(query_x)
torch.cuda.synchronize(); predict_seconds = time.monotonic()-before
columns = np.asarray(model.classes_, dtype=int)
np.testing.assert_array_equal(np.sort(columns), np.arange(18))
proba = np.zeros((len(query_x), 18), dtype=np.float64)
proba[:, columns] = raw
assert np.isfinite(proba).all() and (proba >= 0).all()
np.testing.assert_allclose(proba.sum(1), 1, rtol=0, atol=1e-5)
np.savez(cache/'tabicl_proba.npz', proba=proba, ids=ids, labels=labels)
Path('/kaggle/working/tabicl_inference_audit.json').write_text(json.dumps(dict(
    status='COMPLETE',checkpoint_sha256=settings['checkpoint_sha256'],parameters=params,
    resolved_norm_methods=model.ensemble_generator_.norm_methods_,train_rows=len(y),query_rows=len(ids),
    feature_columns=152,source_table_columns=184,unchanged_152_column_prefix=True,
    fit_seconds=fit_seconds,predict_seconds=predict_seconds,
    peak_cuda_allocated_bytes=int(torch.cuda.max_memory_allocated()),
    peak_cuda_reserved_bytes=int(torch.cuda.max_memory_reserved()),
    packages={p:metadata.version(p) for p in ('tabicl','torch','numpy','scikit-learn','einops')},
    probability_transform='none',row_mass_maxabs=float(np.abs(proba.sum(1)-1).max()),
    oof=False,model_context_exported=False),indent=2))


### Sequential offline execution


In [ ]:
%%writefile /kaggle/temp/ntl-v6-pfn184-code/run_candidate.py
"""Offline sequential baseline, fixed tabular members and CPU finalization."""
import hashlib
import json
import os
import re
import subprocess
import sys
import tempfile
from pathlib import Path

assert os.name!='nt' and Path('/kaggle/input').is_dir(),'Kaggle execution only'
here=Path(__file__).resolve().parent
config=json.loads((here/'runtime_config.json').read_text())
assert config['reference_scored_script_version_id']==359152926
assert not config['second_column_balance'] and config['probability_transform']=='none'
matches=list(Path('/kaggle/input').rglob(config['tabpfn']['checkpoint_filename']))
assert len(matches)==1,'Attach the exact reviewed offline tabular asset package'
asset=matches[0].parent
manifest=json.loads((asset/'runtime_manifest.json').read_text())
assert manifest['status']=='CPU_SETUP_COMPLETE_NO_MODEL_EXECUTION'
assert manifest['global_environment_unchanged'] and not manifest['model_fit'] and not manifest['model_predict']
assert manifest['checkpoint_sha256']==config['tabpfn']['checkpoint_sha256']
assert matches[0].stat().st_size==config['tabpfn']['checkpoint_bytes']
with matches[0].open('rb') as stream:
    assert hashlib.file_digest(stream,'sha256').hexdigest()==config['tabpfn']['checkpoint_sha256']
lock=asset/'runtime-addon-lock.txt'
wheels=asset
assert lock.is_file() and any(wheels.glob('*.whl')),'Reviewed offline addon lock/wheels are required'
for item in manifest['selected_addons'].values():
    with (wheels/item['wheel']).open('rb') as stream:
        assert hashlib.file_digest(stream,'sha256').hexdigest()==item['sha256']
for line in lock.read_text().splitlines():
    if '==' in line and not line.lstrip().startswith('#'):
        package=re.split(r'==',line.strip(),maxsplit=1)[0].lower().replace('_','-')
        assert package not in ('torch','transformers','peft','accelerate','safetensors')
        assert not package.startswith('nvidia-'),'Reuse the original CUDA/Torch runtime'
Path('/kaggle/temp').mkdir(parents=True,exist_ok=True)
temp=Path(tempfile.mkdtemp(prefix='ntl-tabular-',dir='/kaggle/temp'))
site=temp/'site'
base_env=os.environ.copy()
base_env['NTL_TABULAR_TEMP']=str(temp)
base_env['NTL_TABULAR_ASSET']=str(asset)
icl_matches=list(Path('/kaggle/input').rglob(config['tabicl']['checkpoint_filename']))
assert len(icl_matches)==1,'Attach the exact public TabICL offline asset'
icl_asset=icl_matches[0].parent
assert icl_matches[0].stat().st_size==config['tabicl']['checkpoint_bytes']
with icl_matches[0].open('rb') as stream:
    assert hashlib.file_digest(stream,'sha256').hexdigest()==config['tabicl']['checkpoint_sha256']
icl_lock=icl_asset/'tabicl-addon-lock.txt'
assert icl_lock.is_file()
assert icl_lock.read_text().strip()==('tabicl==2.2.0 --hash=sha256:'+config['tabicl']['wheel_sha256'])
icl_wheel=icl_asset/config['tabicl']['wheel_filename']
with icl_wheel.open('rb') as stream:
    assert hashlib.file_digest(stream,'sha256').hexdigest()==config['tabicl']['wheel_sha256']
for license_name in ('TabICL_CODE_LICENSE','MODEL_WEIGHT_LICENSE_DECLARATION.txt','model_license_provenance.json'):
    assert (icl_asset/license_name).is_file()
base_env['NTL_TABICL_ASSET']=str(icl_asset)
subprocess.run([sys.executable,'-m','pip','install','--no-index','--find-links',str(wheels),
    '--no-deps','--require-hashes','--target',str(site),'-r',str(lock)],
    check=True,timeout=config['package_install_timeout_seconds'])
subprocess.run([sys.executable,'-m','pip','install','--no-index','--find-links',str(icl_asset),
    '--no-deps','--require-hashes','--target',str(site),'-r',str(icl_lock)],
    check=True,timeout=config['package_install_timeout_seconds'])
tabular_env=base_env.copy()
tabular_env['PYTHONPATH']=str(site)
tabular_env['HF_HUB_OFFLINE']='1'
subprocess.run([sys.executable,'-c',
    "import json, importlib.metadata as m; from pathlib import Path; import numpy, pandas, scipy, sklearn, torch; from tabpfn import TabPFNClassifier; from tabpfn.constants import ModelVersion; actual=dict(numpy=numpy.__version__, pandas=pandas.__version__, scipy=scipy.__version__, sklearn=sklearn.__version__, torch=torch.__version__, tabpfn=m.version('tabpfn')); expected="+repr(config['verified_runtime_imports'])+"; assert actual==expected, (actual,expected); assert ModelVersion.V3_5; Path('/kaggle/working/tabular_environment_audit.json').write_text(json.dumps(dict(status='VERIFIED_OFFLINE_RUNTIME_IMPORTS',imports=actual,diagnostic_script_version_id=359162368),indent=2)); print('Offline TabPFN imports passed', flush=True)"],
    env=tabular_env,check=True,timeout=90)
subprocess.run([sys.executable,'-c',
    "import json, importlib.metadata as m; from pathlib import Path; from tabicl import TabICLClassifier; import einops; actual=dict(tabicl=m.version('tabicl'),einops=einops.__version__); expected=dict(tabicl='2.2.0',einops='0.8.2'); assert actual==expected,(actual,expected); Path('/kaggle/working/tabicl_environment_audit.json').write_text(json.dumps(dict(status='VERIFIED_OFFLINE_TABICL_IMPORTS_NO_MODEL',imports=actual,isolated_site=True,model_initialized=False),indent=2)); print('Offline TabICL imports passed',flush=True)"],
    env=tabular_env,check=True,timeout=90)
subprocess.run([sys.executable,'-c',
    "import json, importlib.metadata as m; from pathlib import Path; from tabpfn import TabPFNClassifier; from tabicl import TabICLClassifier; import einops,torch,numpy,sklearn; actual=dict(tabpfn=m.version('tabpfn'),tabicl=m.version('tabicl'),einops=einops.__version__,torch=torch.__version__,numpy=numpy.__version__,sklearn=sklearn.__version__); expected=dict(tabpfn='9.0.0',tabicl='2.2.0',einops='0.8.2',torch='2.11.0+cu128',numpy='2.1.3',sklearn='1.6.1'); assert actual==expected,(actual,expected); Path('/kaggle/working/tabular_joint_environment_audit.json').write_text(json.dumps(dict(status='VERIFIED_BOTH_TABULAR_IMPORTS_ONE_CHILD_NO_MODELS',imports=actual,shared_target_directory=True,models_initialized=0),indent=2)); print('Both offline tabular imports passed together',flush=True)"],
    env=tabular_env,check=True,timeout=90)
subprocess.run([sys.executable,str(here/'run_baseline.py')],cwd='/kaggle/working',env=base_env,
    check=True,timeout=config['baseline_timeout_seconds'])
baseline=Path('/kaggle/working/submission.parquet')
assert baseline.is_file()
control=Path('/kaggle/working/ntl_v6_gated_control.parquet')
assert not control.exists()
baseline.rename(control)
subprocess.run([sys.executable,str(here/'tabular_worker.py'),'prepare'],env=tabular_env,
    check=True,timeout=config['prepare_timeout_seconds'])
subprocess.run([sys.executable,str(here/'tabular_worker.py'),'tabpfn'],env=tabular_env,
    check=True,timeout=config['model_timeout_seconds'])
subprocess.run([sys.executable,str(here/'tabicl_worker.py')],env=tabular_env,
    check=True,timeout=config['tabicl_timeout_seconds'])
subprocess.run([sys.executable,str(here/'finish_prediction.py')],env=base_env,check=True,timeout=120)
assert Path('/kaggle/working/submission.parquet').is_file()
assert json.loads(Path('/kaggle/working/tabular_combination_audit.json').read_text())['status']=='NTL_V6_GEOMETRIC_PFN_TABICL_POST_AND_FINAL_CLAUDE_GATE_COMPLETE'


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '/kaggle/temp/ntl-v6-pfn184-code/run_candidate.py'], check=True, timeout=18000)
